# Antibot Challenge

В этой работе решается задача бинарной классификации: для каждого `cookie_id` необходимо определить, является ли пользователь ботом или обычным пользователем. Основная метрика  — **Precision при Recall ≥ 0.70**.

## Используемые библиотеки

В работе используются следующие библиотеки:

* **pandas** и **numpy** — загрузка, очистка и обработка данных;
* **scipy** — дополнительные статистические вычисления;
* **LightGBM** — основная модель градиентного бустинга;
* **CatBoost** — дополнительная модель для сравнения;
* **XGBoost** — ещё один независимый эксперт;
* стандартные библиотеки Python (`pathlib`, `random`, `json` и др.) — работа с файлами, настройками и воспроизводимостью.

## Идея решения

Основная идея состоит в том, чтобы представить историю событий каждого `cookie_id` набором агрегированных признаков. Используются несколько типов информации:

* количество и разнообразие событий;
* интервалы между событиями и скорость активности;
* последовательности событий и повторяющиеся серии;
* pointer-признаки;
* информация об объектах, поисковых запросах и User-Agent;
* активность других cookies с теми же объектами и запросами;
* историческая доля ботов для ранее встречавшихся объектов и других сущностей;
* признаки сессий и концентрации активности во времени.

Особое внимание уделяется временной структуре данных. При построении исторических признаков используются только события из прошлого, поэтому информация из будущего не попадает в признаки текущего пользователя.

Для проверки качества используется **temporal validation**: модель обучается на более ранних днях и проверяется на более поздних. Последний временной блок используется отдельно как `audit` и не участвует в подборе гиперпараметров и весов ансамбля.

В финале несколько моделей объединяются в **rank-space blend**. Для уменьшения зависимости от случайности итоговые модели обучаются с фиксированными seed, после чего их предсказания усредняются и объединяются в один итоговый score.


## 0. Импорты и настройки

Здесь фиксируются random seed, параметры временной валидации, настройки исторических признаков, набор финальных seed-ов и параметры поиска ансамбля.

In [2]:
!pip install -q catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 10.6 MB/s eta 0:00:00


In [3]:
from __future__ import annotations

import gc
import importlib.util
import json
import os
import random
import warnings
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
from scipy.stats import entropy, rankdata
import lightgbm as lgb
from catboost import CatBoostClassifier
import xgboost as xgb

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 150)
pd.set_option("display.width", 180)


# Основные настройки эксперимента
SEED = 42
FINAL_SEEDS = [42, 123, 777]
RECALL_TARGET = 0.70

# Псевдоразметка в финальной версии не используется
USE_PSEUDO_LABELS = False

# Настройки исторических и cross-cookie признаков
HISTORY_ALPHA = 2.0
HISTORY_KEYS = ["item_id", "search_query", "user_agent", "item_category", "item_location"]
CROSS_COOKIE_KEYS = ["item_id", "search_query", "user_agent"]
SESSION_GAP_SEC = 30 * 60

# Дополнительное взвешивание последних дней отключено
FINAL_RECENCY_MODE = "none"

PSEUDO_BOT_THRESHOLD = 0.985
PSEUDO_HUMAN_THRESHOLD = 0.008
PSEUDO_SAMPLE_WEIGHT = 0.25

# Параметры небольшого поиска гиперпараметров
N_LGB_TRIALS = 6
N_CB_TRIALS = 6
N_XGB_TRIALS = 2
EARLY_STOPPING = 120

# Параметры поиска весов итогового ансамбля
BLEND_STEP_COARSE = 0.10
BLEND_STEP_FINE = 0.02
POWER_GRID = [0.85, 1.00, 1.15]

ARTIFACTS_DIR = Path("artifacts_v14_1")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)


# Фиксируем случайность для воспроизводимости результата
def seed_everything(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


seed_everything()

## 1. Пути к данным и метрика

Определяем расположение входных файлов и подключаем официальную `metric.py`. Если её нет, используется локальная реализация той же метрики.

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
!unzip -q /content/drive/MyDrive/avito/bot_detection_challenge.zip -d /content/

In [6]:
def first_existing(paths: Sequence[Path]) -> Optional[Path]:
    for p in paths:
        if p.exists():
            return p
    return None

DATA_DIR = first_existing([
    Path("data"),
    Path("/content/data")
])


TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
EVENTS_PATH = DATA_DIR / "events.csv.gz"
SAMPLE_SUB_PATH = DATA_DIR / "sample_submission.csv"

METRIC_PATH = Path("metric.py")

spec = importlib.util.spec_from_file_location("metric", METRIC_PATH)
metric_mod = importlib.util.module_from_spec(spec)

assert spec.loader is not None
spec.loader.exec_module(metric_mod)

OFFICIAL_METRIC = metric_mod.precision_at_recall

def score_pr(y_true, pred) -> float:
    return float(
        OFFICIAL_METRIC(
            y_true,
            pred,
            recall=RECALL_TARGET
        )
    )

## 2. Загрузка данных

Загружаются `train`, `test` и таблица событий. Для событий сразу задаются типы данных, а временные поля читаются как datetime.

In [7]:
train_meta = pd.read_csv(
    TRAIN_PATH,
    parse_dates=["cookie_created_at", "window_start_ts", "window_end_ts"],
)
test_meta = pd.read_csv(
    TEST_PATH,
    parse_dates=["cookie_created_at", "window_start_ts", "window_end_ts"],
)

y_train = train_meta["target"].astype(int).to_numpy()

EVENTS_DTYPES = {
    "cookie_id": "string",
    "eid": "Int32",
    "event_name": "category",
    "platform": "category",
    "user_agent": "string",
    "item_id": "Int64",
    "item_category": "category",
    "item_location": "category",
    "seller_type": "category",
    "search_query": "string",
    "search_page": "Int32",
    "pointer_x": "float32",
    "pointer_y": "float32",
}

events_raw = pd.read_csv(
    EVENTS_PATH,
    compression="gzip",
    dtype=EVENTS_DTYPES,
    parse_dates=["event_ts"],
)

print(f"train : {train_meta.shape}")
print(f"test  : {test_meta.shape}")
print(f"events: {events_raw.shape}")
print(f"train bot share: {y_train.mean():.4f}")

assert train_meta["cookie_id"].is_unique
assert test_meta["cookie_id"].is_unique
assert set(train_meta["cookie_id"]).isdisjoint(set(test_meta["cookie_id"]))

train : (11091, 5)
test  : (4909, 4)
events: (328905, 14)
train bot share: 0.0811


## 3. Очистка событий

На этом этапе приводим исходные события к единому виду и оставляем только те данные, которые действительно относятся к рассматриваемому `cookie`.

Сначала нормализуется поле `platform`: разные варианты записи одной и той же платформы (`web`, `WEB`, `desktop`, `iOS`, `iphone` и т.д.) объединяются в несколько общих категорий. Это уменьшает количество дублирующих значений и позволяет модели работать с платформами более последовательно.

Далее таблица событий объединяется с информацией из `train.csv` и `test.csv` по `cookie_id`. Благодаря этому для каждого события становятся известны начало и конец временного окна соответствующего cookie. После этого удаляются события, которые произошли за пределами этого окна. Такие записи не относятся к поведению пользователя в рассматриваемый период и поэтому не должны использоваться при построении признаков.

Также удаляются только полностью совпадающие записи событий. Мы специально не используем более агрессивное удаление по отдельным полям, поскольку два похожих события могут на самом деле быть разными действиями пользователя. Такой подход позволяет убрать технические дубликаты, не потеряв полезную информацию о поведении.

После очистки события сортируются по `cookie_id` и времени `event_ts`. Это необходимо для дальнейшего построения временных и последовательностных признаков: интервалов между действиями, серий событий, сессий и других характеристик поведения.

Такая предварительная обработка помогает избежать нескольких проблем: в признаки не попадает активность вне нужного временного окна, технические дубликаты не искажают статистику, а единый формат категориальных значений делает дальнейшее обучение моделей более стабильным.


In [8]:
# Нормализуем платформы
platform_map = {
    "web": "web", "WEB": "web", "Web": "web", "desktop": "web",
    "android": "android", "ANDROID": "android", "Android": "android",
    "ios": "ios", "IOS": "ios", "iOS": "ios", "iphone": "ios",
}

events_raw["platform"] = (
    events_raw["platform"].astype("string").map(platform_map)
    .fillna(events_raw["platform"].astype("string"))
    .astype("category")
)

# Добавляем временные окна
meta_all = pd.concat(
    [
        train_meta[["cookie_id", "window_start_ts", "window_end_ts"]],
        test_meta[["cookie_id", "window_start_ts", "window_end_ts"]],
    ],
    ignore_index=True,
).drop_duplicates("cookie_id")

events = events_raw.merge(meta_all, on="cookie_id", how="inner")

# Оставляем события внутри окна
in_window = (
    (events["event_ts"] >= events["window_start_ts"])
    & (events["event_ts"] < events["window_end_ts"])
)

print(f"Событий после join: {len(events):,}")
print(f"Событий внутри окна: {int(in_window.sum()):,} ({100*in_window.mean():.2f}%)")

events = events.loc[in_window].copy()
events = events.drop(columns=["window_start_ts", "window_end_ts"])

# Удаляем дубликаты
WORKING_DUP_COLS = [c for c in events.columns if c not in []]

before = len(events)
events = events.drop_duplicates(subset=WORKING_DUP_COLS, keep="first")

print(f"Удалено полных технических дублей: {before - len(events):,}")

# Сортируем по времени
events = events.sort_values(
    ["cookie_id", "event_ts"],
    kind="mergesort"
).reset_index(drop=True)

Событий после join: 328,905
Событий внутри окна: 288,126 (87.60%)
Удалено полных технических дублей: 4,296


## 4. Признаки отдельных событий

На этом этапе для каждого события отдельно строятся признаки, которые описывают скорость действий пользователя, особенности браузера и движение указателя. Идея в том, чтобы получить более подробное описание поведения, а уже потом агрегировать его на уровне `cookie_id`.

### Временные признаки

Сначала считается время от предыдущего события пользователя — `diff_ms`. По нему дополнительно создаются признаки:

* `is_diff_zero` — события произошли в один и тот же момент;
* `is_fast_100ms`, `is_fast_500ms`, `is_fast_1s` — очень маленький интервал между событиями;
* `is_pause_20_50ms` — небольшая пауза;
* `is_burst_20ms`, `is_burst_50ms` — события идут практически подряд.

Эти признаки нужны, чтобы увидеть слишком быстрые или повторяющиеся действия. Для бота такой шаблон поведения встречается чаще, чем у обычного пользователя.

Также сравниваются текущий и предыдущий `eid` и `event_name`. Признаки `same_eid_as_prev` и `same_event_as_prev` показывают, насколько часто пользователь повторяет одно и то же действие подряд.

### User-Agent

Для `user_agent` вычисляется его длина и несколько простых индикаторов:

* `ua_len` — длина User-Agent;
* `ua_is_bot_keyword` — наличие характерных для автоматических инструментов строк, например `selenium`, `headless`, `python-requests`, `curl`;
* `ua_is_app` — признак использования приложения.

Это помогает учитывать особенности среды, из которой отправляются события. Некоторые варианты User-Agent могут быть связаны с автоматизированными запросами.

### Пропуски

Для основных полей создаются бинарные признаки `miss_*`, которые показывают, отсутствовало ли значение.

Например, отдельно проверяются пропуски в `user_agent`, `item_id`, `search_query`, `search_page`, `pointer_x` и других полях.

Пропуски могут быть полезны сами по себе, потому что разные типы действий и разные способы взаимодействия с сайтом могут иметь различную структуру заполнения данных.

### Pointer-признаки

Для событий, где есть координаты указателя, рассчитываются:

* `pointer_dist` — расстояние между текущей и предыдущей позицией;
* `pointer_speed` — скорость движения;
* `pointer_accel` — изменение скорости;
* `pointer_abs_accel` — абсолютное ускорение;
* `pointer_zero_move` — отсутствие движения;
* `has_pointer` — наличие координат указателя.

Эти признаки позволяют учитывать характер движения пользователя. Реальное взаимодействие с интерфейсом обычно отличается от полностью автоматизированных действий по времени и движению указателя.

### Pagination

Для `search_page` считаются изменения номера страницы:

* `search_page_diff`;
* `search_page_back` — переход назад;
* `search_page_forward` — переход вперёд.

Это позволяет учитывать последовательность просмотра результатов поиска и различать разные сценарии поведения.

### Время события

Для каждого события также сохраняются:

* `event_hour` — час события;
* `event_dow` — день недели;
* `is_weekend` — выходной день;
* `is_night` — ночное время;
* `hour_sin` и `hour_cos` — циклическое представление часа.

Такие признаки позволяют модели учитывать временные особенности активности пользователей. Например, поведение в ночное время или различия между будними и выходными днями могут быть полезны при классификации.

Все эти признаки сначала строятся на уровне отдельных событий. В следующих этапах они агрегируются по `cookie_id`, чтобы получить уже общее описание поведения каждого пользователя.

In [9]:
# Предыдущее событие для каждого cookie
G_COOKIE = events.groupby("cookie_id", observed=True)

events["prev_ts"] = G_COOKIE["event_ts"].shift(1)

# Интервалы и слишком быстрые действия
events["diff_ms"] = (
    events["event_ts"] - events["prev_ts"]
).dt.total_seconds() * 1000.0

events["is_diff_zero"] = (events["diff_ms"] == 0).astype("int8")
events["is_fast_100ms"] = (events["diff_ms"] < 100).astype("int8")
events["is_fast_500ms"] = (events["diff_ms"] < 500).astype("int8")
events["is_fast_1s"] = (events["diff_ms"] < 1000).astype("int8")
events["is_pause_20_50ms"] = (
    events["diff_ms"].between(20, 50, inclusive="both")
).astype("int8")
events["is_burst_20ms"] = (events["diff_ms"] <= 20).astype("int8")
events["is_burst_50ms"] = (events["diff_ms"] <= 50).astype("int8")

# Повторяющиеся действия
events["same_eid_as_prev"] = (
    events["eid"] == G_COOKIE["eid"].shift(1)
).fillna(False).astype("int8")

events["same_event_as_prev"] = (
    events["event_name"] == G_COOKIE["event_name"].shift(1)
).fillna(False).astype("int8")


# User-Agent
ua = events["user_agent"].astype("string").fillna("")

events["ua_len"] = ua.str.len().astype("float32")

events["ua_is_bot_keyword"] = ua.str.lower().str.contains(
    r"headless|python-requests|scrapy|selenium|puppeteer|phantomjs|curl/|wget/|go-http-client|okhttp|axios/",
    regex=True,
    na=False,
).astype("int8")

events["ua_is_app"] = ua.str.startswith("Avito/").astype("int8")


# Пропуски в основных полях
for c in [
    "user_agent", "item_id", "item_category", "item_location",
    "seller_type", "search_query", "search_page", "pointer_x"
]:
    if c in events.columns:
        events[f"miss_{c}"] = events[c].isna().astype("int8")


# Движение указателя
events["prev_x"] = G_COOKIE["pointer_x"].shift(1)
events["prev_y"] = G_COOKIE["pointer_y"].shift(1)

events["has_pointer"] = (
    events["pointer_x"].notna() & events["pointer_y"].notna()
).astype("int8")

dx = events["pointer_x"] - events["prev_x"]
dy = events["pointer_y"] - events["prev_y"]

events["pointer_dist"] = np.sqrt(dx * dx + dy * dy)
events["pointer_speed"] = (
    events["pointer_dist"] / events["diff_ms"].clip(lower=1.0)
)

events.loc[events["diff_ms"].isna(), "pointer_speed"] = np.nan

events["pointer_zero_move"] = (
    (events["has_pointer"] == 1) &
    (events["pointer_dist"] == 0)
).astype("int8")

events["prev_speed"] = G_COOKIE["pointer_speed"].shift(1)
delta_sec = (events["diff_ms"] / 1000.0).clip(lower=0.001)

events["pointer_accel"] = (
    (events["pointer_speed"] - events["prev_speed"]) / delta_sec
)

events["pointer_abs_accel"] = events["pointer_accel"].abs()


# Переходы между страницами поиска
if "search_page" in events.columns:
    events["prev_search_page"] = G_COOKIE["search_page"].shift(1)

    events["search_page_diff"] = (
        events["search_page"] - events["prev_search_page"]
    )

    events["search_page_back"] = (
        events["search_page_diff"] < 0
    ).fillna(False).astype("int8")

    events["search_page_forward"] = (
        events["search_page_diff"] > 0
    ).fillna(False).astype("int8")


# Базовые временные признаки
events["event_hour"] = events["event_ts"].dt.hour.astype("int8")
events["event_dow"] = events["event_ts"].dt.dayofweek.astype("int8")

events["is_weekend"] = (
    events["event_dow"] >= 5
).astype("int8")

events["is_night"] = (
    events["event_hour"].between(0, 5)
).astype("int8")

# Циклическое представление часа
events["hour_sin"] = np.sin(
    2 * np.pi * events["event_hour"] / 24.0
)

events["hour_cos"] = np.cos(
    2 * np.pi * events["event_hour"] / 24.0
)

## 5. Последовательности, burst и sessions

Здесь смотрИМ уже не на отдельные события, а на то, как пользователь действует в целом и в каком порядке происходят события.

Сначала считаются серии одинаковых событий и eid. Например, можно увидеть, сколько одинаковых действий подряд сделал пользователь. Это помогает заметить повторяющиеся и слишком однообразные сценарии.

Дальше строятся burst-признаки для промежутков в 1, 5 и 30 секунд. Они показывают, насколько долго пользователь может выполнять большое количество действий практически без пауз.

Также учитываются переходы между событиями. Для каждой пары последовательных действий сохраняется их комбинация, например event_A -> event_B. Это позволяет модели учитывать не только сами события, но и типичную последовательность действий.

История пользователя дополнительно разбивается на сессии. Новая сессия начинается, если между событиями прошло больше 30 минут. Для сессий считаются их количество и положение событий внутри них.

В конце определяется положение события в истории cookie: относится ли оно к первым или последним 10% и 25% событий. Это помогает учитывать, как именно распределяется активность пользователя во времени.

В итоге этот блок нужен, чтобы модель видела не только сколько действий совершил пользователь, но и характер их последовательности и интенсивности.

In [10]:
for col, prefix in [("event_name", "event_run"), ("eid", "eid_run")]:
    changed = (
        events[col] != G_COOKIE[col].shift(1)
    ).fillna(True).astype("int8")
    run_id = changed.groupby(events["cookie_id"], observed=True).cumsum()
    events[f"{prefix}_pos"] = (
        events.groupby(["cookie_id", run_id], observed=True).cumcount() + 1
    ).astype("int32")

for threshold_ms, prefix in [(1000, "burst_1s"), (5000, "burst_5s"), (30000, "burst_30s")]:
    start = (events["diff_ms"].isna() | (events["diff_ms"] > threshold_ms)).astype("int8")
    run_id = start.groupby(events["cookie_id"], observed=True).cumsum()
    events[f"{prefix}_pos"] = (
        events.groupby(["cookie_id", run_id], observed=True).cumcount() + 1
    ).astype("int32")
    events[f"{prefix}_start"] = start

events["same_second_event_count"] = (
    events.groupby(["cookie_id", "event_ts"], observed=True)["eid"].transform("size")
)

prev_name = G_COOKIE["event_name"].shift(1).astype("string")
cur_name = events["event_name"].astype("string")
events["event_transition"] = prev_name.fillna("__START__") + "__TO__" + cur_name.fillna("__NA__")
events.loc[prev_name.isna(), "event_transition"] = "__START__"

events["event_transition_repeat"] = (
    events["event_transition"].eq(
        events.groupby("cookie_id", observed=True)["event_transition"].shift(1)
    )
).fillna(False).astype("int8")

events["session_start"] = (
    events["diff_ms"].isna() | (events["diff_ms"] > SESSION_GAP_SEC * 1000.0)
).astype("int8")
events["session_id"] = events.groupby("cookie_id", observed=True)["session_start"].cumsum().astype("int32")
events["session_pos"] = (
    events.groupby(["cookie_id", "session_id"], observed=True).cumcount() + 1
).astype("int32")

events["cookie_event_idx"] = events.groupby("cookie_id", observed=True).cumcount().astype("int32")
events["cookie_event_n"] = events.groupby("cookie_id", observed=True)["eid"].transform("size").astype("int32")
events["first10_event"] = (events["cookie_event_idx"] < np.ceil(events["cookie_event_n"] * 0.10)).astype("int8")
events["first25_event"] = (events["cookie_event_idx"] < np.ceil(events["cookie_event_n"] * 0.25)).astype("int8")
events["last10_event"] = (events["cookie_event_idx"] >= np.floor(events["cookie_event_n"] * 0.90)).astype("int8")
events["last25_event"] = (events["cookie_event_idx"] >= np.floor(events["cookie_event_n"] * 0.75)).astype("int8")


### 6. Strict-past relational признаки

На этом этапе добавляются признаки, которые учитывают не только действия самого cookie, но и предыдущую активность других пользователей.

Для item_id и search_query считаются:

reach — сколько разных cookies уже использовали этот объект или запрос;
prior_events — сколько таких событий уже было;
gap_sec — сколько времени прошло с предыдущего события;
first_seen_flag — первое ли это появление.

Для user_agent также считается предыдущий reach.

Дополнительно используются cross-cookie признаки: насколько недавно тот же item, запрос или User-Agent встречался у другого cookie. Для них добавляются пороги <1s, <5s и <30s.

Также считается reach для категории и локации объекта.

Все эти признаки используют только строго прошлые события, поэтому информация из будущего не попадает в модель. Это помогает учитывать общую активность пользователей и обнаруживать необычно плотные или повторяющиеся сценарии поведения.

In [11]:
def q10(s):
    return s.quantile(0.10)


def q25(s):
    return s.quantile(0.25)


def q75(s):
    return s.quantile(0.75)


def q90(s):
    return s.quantile(0.90)


def q95(s):
    return s.quantile(0.95)


def iqr(s):
    return s.quantile(0.75) - s.quantile(0.25)


def safe_entropy(s):
    v = s.dropna().value_counts()
    return float(entropy(v)) if len(v) else 0.0


def strict_prior_unique(df: pd.DataFrame, key_col: str) -> np.ndarray:
    work = df[["cookie_id", "event_ts", key_col]].copy()
    work["_row"] = np.arange(len(work), dtype=np.int64)
    valid = work[key_col].notna().to_numpy()
    out = np.full(len(work), np.nan, dtype=np.float32)

    pairs = work.loc[valid, [key_col, "cookie_id", "event_ts"]]
    first_seen = (
        pairs.groupby([key_col, "cookie_id"], observed=True)["event_ts"]
        .min()
        .reset_index(name="first_seen_ts")
    )

    first_seen["ts_i64"] = first_seen["first_seen_ts"].astype("int64")
    time_counts = (
        first_seen.groupby([key_col, "ts_i64"], observed=True)
        .size()
        .rename("n_new")
        .reset_index()
        .sort_values([key_col, "ts_i64"], kind="mergesort")
    )
    time_counts["cum"] = time_counts.groupby(key_col, observed=True)["n_new"].cumsum()
    time_counts["reach_before"] = time_counts["cum"] - time_counts["n_new"]

    left_groups = work.loc[valid].groupby(key_col, observed=True).groups
    right_groups = time_counts.groupby(key_col, observed=True).groups

    for key_value, row_index in left_groups.items():
        right_index = right_groups.get(key_value)
        if right_index is None:
            out[np.asarray(row_index, dtype=np.int64)] = 0.0
            continue

        ri = np.asarray(right_index, dtype=np.int64)
        # time_counts is globally sorted by key then ts, so ri is already sorted by ts.
        right_ts = time_counts.iloc[ri]["ts_i64"].to_numpy(dtype=np.int64)
        right_reach = time_counts.iloc[ri]["reach_before"].to_numpy(dtype=np.float32)

        li = np.asarray(row_index, dtype=np.int64)
        event_ts = work.iloc[li]["event_ts"].astype("int64").to_numpy()
        pos = np.searchsorted(right_ts, event_ts, side="left")
        vals = np.zeros(len(pos), dtype=np.float32)
        nonzero = pos > 0
        vals[nonzero] = right_reach[pos[nonzero] - 1]
        out[li] = vals

    return out


def prior_event_count_and_recency(df: pd.DataFrame, key_col: str, prefix: str):
    valid = df[key_col].notna()
    tmp = df[[key_col, "event_ts"]].copy()
    tmp["_row"] = np.arange(len(tmp), dtype=np.int64)
    tmp["_valid"] = valid.to_numpy()
    tmp_valid = tmp.loc[valid].sort_values([key_col, "event_ts", "_row"], kind="mergesort")
    tmp_valid[f"{prefix}_prior_events"] = tmp_valid.groupby(key_col, observed=True).cumcount().astype("float32")
    tmp_valid[f"{prefix}_prev_ts"] = tmp_valid.groupby(key_col, observed=True)["event_ts"].shift(1)
    tmp_valid[f"{prefix}_gap_sec"] = (
        tmp_valid["event_ts"] - tmp_valid[f"{prefix}_prev_ts"]
    ).dt.total_seconds()
    tmp_valid[f"{prefix}_first_flag"] = (tmp_valid[f"{prefix}_prior_events"] == 0).astype("int8")

    result = tmp[["_row"]].merge(
        tmp_valid[["_row", f"{prefix}_prior_events", f"{prefix}_gap_sec", f"{prefix}_first_flag"]],
        on="_row", how="left"
    )
    return (
        result[f"{prefix}_prior_events"].to_numpy(),
        result[f"{prefix}_gap_sec"].to_numpy(),
        result[f"{prefix}_first_flag"].to_numpy(),
    )


for key_col, prefix in [("item_id", "item"), ("search_query", "query")]:
    events[f"{prefix}_reach"] = strict_prior_unique(events, key_col)
    prior_n, prev_gap, first_flag = prior_event_count_and_recency(events, key_col, prefix)
    events[f"{prefix}_prior_events"] = prior_n
    events[f"{prefix}_gap_sec"] = prev_gap
    events[f"{prefix}_first_seen_flag"] = first_flag

events["ua_reach"] = strict_prior_unique(events, "user_agent")


def strict_prev_other_cookie_gap(df: pd.DataFrame, key_col: str) -> np.ndarray:
    work = df[["cookie_id", "event_ts", key_col]].copy()
    work["_row"] = np.arange(len(work), dtype=np.int64)
    valid = work[key_col].notna()
    out = np.full(len(work), np.nan, dtype=np.float32)
    w = work.loc[valid].sort_values([key_col, "event_ts", "cookie_id", "_row"], kind="mergesort")

    for key_value, idx in w.groupby(key_col, observed=True, sort=False).groups.items():
        sub = w.loc[idx, ["_row", "cookie_id", "event_ts"]].sort_values(
            ["event_ts", "cookie_id", "_row"], kind="mergesort"
        )
        rows = sub["_row"].to_numpy(dtype=np.int64)
        cookies = sub["cookie_id"].astype("string").to_numpy()
        ts = sub["event_ts"].astype("int64").to_numpy()
        gaps = np.full(len(sub), np.nan, dtype=np.float32)

        best_time = None
        best_cookie = None
        second_time = None
        second_cookie = None
        i = 0
        while i < len(sub):
            j = i + 1
            while j < len(sub) and ts[j] == ts[i]:
                j += 1
            current_ts = ts[i]
            for k in range(i, j):
                if best_cookie is not None:
                    prior_ts = best_time if cookies[k] != best_cookie else second_time
                    if prior_ts is not None:
                        gaps[k] = (current_ts - prior_ts) / 1e9

            group_cookie_times = {}
            for k in range(i, j):
                group_cookie_times[cookies[k]] = current_ts
            for gcookie in group_cookie_times:
                gtime = group_cookie_times[gcookie]
                if best_cookie is None or gtime > best_time:
                    if best_cookie != gcookie:
                        second_cookie, second_time = best_cookie, best_time
                    best_cookie, best_time = gcookie, gtime
                elif gcookie != best_cookie and (second_time is None or gtime > second_time):
                    second_cookie, second_time = gcookie, gtime
            i = j

        out[rows] = gaps

    return out

for key_col, prefix in [("item_id", "item"), ("search_query", "query"), ("user_agent", "ua")]:
    events[f"{prefix}_cross_cookie_gap_sec"] = strict_prev_other_cookie_gap(events, key_col)
    for threshold in [1.0, 5.0, 30.0]:
        events[f"{prefix}_cross_cookie_lt_{int(threshold)}s"] = (
            events[f"{prefix}_cross_cookie_gap_sec"] < threshold
        ).fillna(False).astype("int8")


for key_col, prefix in [("item_category", "category"), ("item_location", "location")]:
    events[f"{prefix}_reach"] = strict_prior_unique(events, key_col)

for c in [
    "item_reach", "query_reach", "ua_reach",
    "item_prior_events", "query_prior_events",
]:
    events[f"log1p_{c}"] = np.log1p(events[c].clip(lower=0))


## 7. Агрегация до уровня cookie

После построения признаков для отдельных событий собираем их в одну строку для каждого cookie_id.

Для временных и числовых признаков считаются средние, минимальные и максимальные значения, медианы, стандартные отклонения и квантили. Для событий отдельно считаются количества разных event_name и переходов между событиями.

Также добавляются общие характеристики пользователя: активные часы и дни, наиболее частая платформа, переключения User-Agent и платформы.

Отдельно считаются temporal concentration — максимальное количество событий за 1, 5, 15 и 30 минут — и характеристики сессий: их количество, размер и продолжительность.

В результате получается одна итоговая таблица признаков, где каждая строка соответствует одному cookie.

In [12]:
# Основные статистики по событиям для каждого cookie
AGG_RULES: Dict[str, List[object]] = {
    "event_ts": ["count", "min", "max"],
    "eid": ["nunique"],
    "event_name": ["nunique"],
    "platform": ["nunique"],
    "user_agent": ["nunique"],
    "ua_len": ["mean", "min", "max", "std"],
    "ua_is_bot_keyword": ["mean", "sum", "max"],
    "ua_is_app": ["mean", "sum"],
    "same_second_event_count": ["mean", "max", "std"],
    "same_eid_as_prev": ["mean", "sum"],
    "same_event_as_prev": ["mean", "sum"],
    "event_transition_repeat": ["mean", "sum"],
    "diff_ms": ["mean", "std", "median", "min", "max", q10, q25, q75, q90, q95, iqr, "skew", safe_entropy],
    "is_diff_zero": ["mean", "sum"],
    "is_fast_100ms": ["mean", "sum"],
    "is_fast_500ms": ["mean", "sum"],
    "is_fast_1s": ["mean", "sum"],
    "is_pause_20_50ms": ["mean", "sum"],
    "is_burst_20ms": ["mean", "sum"],
    "is_burst_50ms": ["mean", "sum"],
    "has_pointer": ["mean", "sum"],
    "pointer_dist": ["mean", "std", "median", "max", "sum", q90],
    "pointer_speed": ["mean", "std", "median", "max", q10, q25, q75, q90],
    "pointer_accel": ["mean", "std", "median", "max"],
    "pointer_abs_accel": ["mean", "std", "median", "max", q90],
    "pointer_zero_move": ["mean", "sum"],
    "item_id": ["nunique"],
    "item_category": ["nunique"],
    "item_location": ["nunique"],
    "seller_type": ["nunique"],
    "search_query": ["nunique"],
    "item_reach": ["mean", "max", q90],
    "query_reach": ["mean", "max", q90],
    "ua_reach": ["max", q90],
    "item_prior_events": ["mean", "max", q90],
    "query_prior_events": ["mean", "max", q90],
    "item_gap_sec": ["median", q90],
    "query_gap_sec": ["median", q90],
    "item_first_seen_flag": ["mean", "sum"],
    "query_first_seen_flag": ["mean", "sum"],
    "log1p_item_reach": ["mean", "max"],
    "log1p_query_reach": ["mean", "max"],
    "log1p_item_prior_events": ["mean", "max"],
    "log1p_query_prior_events": ["mean", "max"],
    "hour_sin": ["mean", "std"],
    "hour_cos": ["mean", "std"],
    "is_night": ["mean", "sum"],
    "is_weekend": ["mean", "sum"],
    "event_run_pos": ["mean", "max"],
    "eid_run_pos": ["mean", "max"],
    "burst_1s_pos": ["mean", "max"],
    "burst_5s_pos": ["mean", "max"],
    "burst_30s_pos": ["mean", "max"],
    "burst_1s_start": ["sum"],
    "burst_5s_start": ["sum"],
    "burst_30s_start": ["sum"],
    "session_start": ["sum"],
    "session_pos": ["mean", "max"],
    "first10_event": ["mean", "sum"],
    "first25_event": ["mean", "sum"],
    "last10_event": ["mean", "sum"],
    "last25_event": ["mean", "sum"],
    "item_cross_cookie_gap_sec": ["median", q90, "min"],
    "query_cross_cookie_gap_sec": ["median", q90, "min"],
    "ua_cross_cookie_gap_sec": ["median", q90, "min"],
    "item_cross_cookie_lt_1s": ["mean", "sum"],
    "item_cross_cookie_lt_5s": ["mean", "sum"],
    "item_cross_cookie_lt_30s": ["mean", "sum"],
    "query_cross_cookie_lt_1s": ["mean", "sum"],
    "query_cross_cookie_lt_5s": ["mean", "sum"],
    "query_cross_cookie_lt_30s": ["mean", "sum"],
    "ua_cross_cookie_lt_1s": ["mean", "sum"],
    "ua_cross_cookie_lt_5s": ["mean", "sum"],
    "ua_cross_cookie_lt_30s": ["mean", "sum"],
    "category_reach": ["mean", "max", q90],
    "location_reach": ["mean", "max", q90],
}

# Добавляем статистики по пропускам
for c in [
    "miss_user_agent", "miss_item_id", "miss_item_category",
    "miss_item_location", "miss_seller_type", "miss_search_query",
    "miss_search_page", "miss_pointer_x",
]:
    if c in events.columns:
        AGG_RULES[c] = ["mean", "sum"]

# Признаки пагинации
if "search_page" in events.columns:
    AGG_RULES.update({
        "search_page": ["max", "mean", "std"],
        "search_page_diff": ["mean", "max", "std"],
        "search_page_back": ["mean", "sum"],
        "search_page_forward": ["mean", "sum"],
    })

# Оставляем только существующие признаки
AGG_RULES = {
    c: rules for c, rules in AGG_RULES.items()
    if c in events.columns
}

# Собираем event-level признаки в одну строку на cookie
events_agg = events.groupby("cookie_id", observed=True).agg(AGG_RULES)

events_agg.columns = [
    f"{c}_{a.__name__ if hasattr(a, '__name__') else a}"
    for c, a in events_agg.columns
]

events_agg = events_agg.reset_index()


# Количество каждого типа события
event_type_counts = (
    events.groupby(["cookie_id", "event_name"], observed=True)
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

event_type_counts.columns = [
    "cookie_id" if c == "cookie_id" else f"cnt_{str(c)}"
    for c in event_type_counts.columns
]


# Считаем переходы между событиями
transition_counts = (
    events.groupby(["cookie_id", "event_transition"], observed=True)
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

transition_counts.columns = [
    "cookie_id" if c == "cookie_id" else f"tr_{str(c)}"
    for c in transition_counts.columns
]


# Основные характеристики поведения cookie
def mode_safe(s):
    m = s.mode(dropna=True)
    return m.iloc[0] if len(m) else np.nan


dominant_platform = (
    events.groupby("cookie_id", observed=True)["platform"]
    .agg(mode_safe)
    .rename("dominant_platform")
    .reset_index()
)

hour_entropy = (
    events.groupby("cookie_id", observed=True)["event_hour"]
    .agg(safe_entropy)
    .rename("event_hour_entropy")
    .reset_index()
)

hour_nunique = (
    events.groupby("cookie_id", observed=True)["event_hour"]
    .nunique()
    .rename("active_hours_nunique")
    .reset_index()
)


# Количество активных дней
_tmp_day = events[["cookie_id", "event_ts"]].copy()
_tmp_day["_day"] = _tmp_day["event_ts"].dt.floor("D")

active_days = (
    _tmp_day.groupby("cookie_id", observed=True)["_day"]
    .nunique()
    .rename("active_days_nunique")
    .reset_index()
)

del _tmp_day


# Сколько раз менялись платформа и User-Agent
platform_switch = (
    events.groupby("cookie_id", observed=True)["platform"]
    .apply(
        lambda s: float(
            (s.astype("string") != s.astype("string").shift(1)).sum() - 1
        ) if len(s) else 0.0
    )
    .rename("platform_switches")
    .reset_index()
)

ua_switch = (
    events.groupby("cookie_id", observed=True)["user_agent"]
    .apply(
        lambda s: float(
            (s.astype("string") != s.astype("string").shift(1)).sum() - 1
        ) if len(s) else 0.0
    )
    .rename("ua_switches")
    .reset_index()
)


# Объединяем дополнительные признаки
extra_cookie = event_type_counts.merge(
    transition_counts, on="cookie_id", how="outer"
)

extra_cookie = extra_cookie.merge(
    dominant_platform, on="cookie_id", how="left"
)
extra_cookie = extra_cookie.merge(
    hour_entropy, on="cookie_id", how="left"
)
extra_cookie = extra_cookie.merge(
    hour_nunique, on="cookie_id", how="left"
)
extra_cookie = extra_cookie.merge(
    active_days, on="cookie_id", how="left"
)
extra_cookie = extra_cookie.merge(
    platform_switch, on="cookie_id", how="left"
)
extra_cookie = extra_cookie.merge(
    ua_switch, on="cookie_id", how="left"
)


# Максимальная концентрация событий за разные промежутки времени
concentration_tables = []

for freq, prefix in [
    ("1min", "conc_1m"),
    ("5min", "conc_5m"),
    ("15min", "conc_15m"),
    ("30min", "conc_30m")
]:
    bucket = events["event_ts"].dt.floor(freq)

    tmp = (
        events.assign(_bucket=bucket)
        .groupby(["cookie_id", "_bucket"], observed=True)
        .size()
        .rename("n")
        .reset_index()
    )

    mx = (
        tmp.groupby("cookie_id", observed=True)["n"]
        .max()
        .rename(f"{prefix}_max_events")
    )

    concentration_tables.append(mx.reset_index())


for t in concentration_tables:
    extra_cookie = extra_cookie.merge(
        t, on="cookie_id", how="outer"
    )


# Характеристики сессий
session_summary = (
    events.groupby(["cookie_id", "session_id"], observed=True)
    .agg(
        session_events=("eid", "size"),
        session_start_ts=("event_ts", "min"),
        session_end_ts=("event_ts", "max"),
    )
    .reset_index()
)

session_summary["session_duration_sec"] = (
    session_summary["session_end_ts"]
    - session_summary["session_start_ts"]
).dt.total_seconds()

session_cookie = (
    session_summary.groupby("cookie_id", observed=True)
    .agg(
        session_count=("session_id", "nunique"),
        session_events_mean=("session_events", "mean"),
        session_events_max=("session_events", "max"),
        session_duration_mean=("session_duration_sec", "mean"),
        session_duration_max=("session_duration_sec", "max"),
    )
    .reset_index()
)

extra_cookie = extra_cookie.merge(
    session_cookie, on="cookie_id", how="outer"
)

## 8. Финальная сборка признаков

На этом этапе все рассчитанные ранее признаки объединяются с метаданными train и test, чтобы получить одну итоговую таблицу на каждый cookie_id. Также здесь создаются несколько дополнительных признаков, которые описывают активность пользователя в целом.

Сначала считаются признаки активности: продолжительность окна, длительность реальной активности, доля окна, в которой пользователь был активен, количество событий в единицу времени и положение первого и последнего события внутри окна.

Затем добавляются различные отношения, например число уникальных товаров к общему числу событий или доля событий с pointer. Это позволяет сравнивать пользователей с разным количеством действий.

Также считаются:

доля активности в коротких временных промежутках;
размер самой большой сессии;
количество сессий в час;
коэффициенты вариации временных интервалов и скорости pointer;
несколько простых признаков поведения, например contact_rate, favorite_rate, наличие captcha и login.

После этого признаки делятся на несколько наборов:

kinematic — в основном временные и pointer-признаки;
extended — временные, поведенческие и relational-признаки;
hybrid — объединяет оба набора.

Перед обучением удаляются признаки, которые имеют только одно значение на train. Это убирает бесполезные столбцы и немного уменьшает размерность данных.

В итоге получаем несколько вариантов набора признаков, которые затем используются разными моделями и сравниваются между собой.

In [13]:
# Служебные колонки, которые не используем как признаки
NON_FEATURE_COLS = {
    "cookie_id", "cookie_created_at", "window_start_ts", "window_end_ts", "target"
}


def assemble_features(meta_df: pd.DataFrame) -> pd.DataFrame:
    # Объединяем метаданные и агрегаты событий
    df = meta_df.merge(events_agg, on="cookie_id", how="left")
    df = df.merge(extra_cookie, on="cookie_id", how="left")

    df["has_events"] = df["event_ts_count"].notna().astype("int8")
    df["event_ts_count"] = df["event_ts_count"].fillna(0)

    # Концентрация активности
    for freq_prefix in ["1m", "5m", "15m", "30m"]:
        max_col = f"conc_{freq_prefix}_max_events"
        if max_col in df.columns:
            df[f"conc_{freq_prefix}_share"] = (
                df[max_col].fillna(0) / (df["event_ts_count"] + 1.0)
            )

    if {"session_events_max", "event_ts_count"}.issubset(df.columns):
        df["largest_session_fraction"] = (
            df["session_events_max"].fillna(0)
            / (df["event_ts_count"] + 1.0)
        )

    # Общие временные характеристики cookie
    df["cookie_age_hours"] = (
        (df["window_end_ts"] - df["cookie_created_at"])
        .dt.total_seconds() / 3600.0
    )

    df["window_duration_hours"] = (
        (df["window_end_ts"] - df["window_start_ts"])
        .dt.total_seconds() / 3600.0
    )

    if "session_count" in df.columns:
        df["sessions_per_hour"] = (
            df["session_count"].fillna(0)
            / df["window_duration_hours"].clip(lower=1e-3)
        )

    active_sec = (
        df["event_ts_max"] - df["event_ts_min"]
    ).dt.total_seconds()

    window_sec = (
        df["window_end_ts"] - df["window_start_ts"]
    ).dt.total_seconds()

    df["active_duration_sec"] = active_sec
    df["active_window_coverage"] = (
        active_sec / window_sec.replace(0, np.nan)
    )

    df["first_event_offset_sec"] = (
        df["event_ts_min"] - df["window_start_ts"]
    ).dt.total_seconds()

    df["last_event_offset_sec"] = (
        df["window_end_ts"] - df["event_ts_max"]
    ).dt.total_seconds()

    df["events_per_active_sec"] = (
        df["event_ts_count"] / (active_sec.fillna(0) + 1.0)
    )

    df["events_per_window_hour"] = (
        df["event_ts_count"]
        / (df["window_duration_hours"] * 3600 + 1.0)
    )

    # Относительные показатели
    for num, den, name in [
        ("item_id_nunique", "event_ts_count", "item_to_event_ratio"),
        ("item_category_nunique", "event_ts_count", "category_to_event_ratio"),
        ("search_query_nunique", "event_ts_count", "query_to_event_ratio"),
        ("user_agent_nunique", "event_ts_count", "ua_to_event_ratio"),
        ("has_pointer_sum", "event_ts_count", "pointer_event_ratio"),
        ("burst_1s_pos_max", "event_ts_count", "burst_1s_event_fraction"),
    ]:
        if num in df.columns and den in df.columns:
            df[name] = df[num] / (df[den] + 1.0)

    if "user_agent_nunique" in df.columns:
        df["ua_rotation"] = (
            df["user_agent_nunique"] > 1
        ).astype("int8")

    # Несколько дополнительных показателей разброса
    if {"diff_ms_std", "diff_ms_mean"}.issubset(df.columns):
        df["diff_ms_cv"] = (
            df["diff_ms_std"]
            / (df["diff_ms_mean"].abs() + 1.0)
        )

    if {"pointer_speed_std", "pointer_speed_mean"}.issubset(df.columns):
        df["pointer_speed_cv"] = (
            df["pointer_speed_std"]
            / (df["pointer_speed_mean"].abs() + 1.0)
        )

    if {"pointer_abs_accel_mean", "pointer_speed_mean"}.issubset(df.columns):
        df["accel_to_speed_ratio"] = (
            df["pointer_abs_accel_mean"]
            / (df["pointer_speed_mean"].abs() + 1.0)
        )

    if {"item_first_seen_flag_mean", "item_id_nunique"}.issubset(df.columns):
        df["item_novelty_per_unique_item"] = (
            df["item_first_seen_flag_mean"]
        )

    # Для cookie без событий счётчики заменяем нулями
    count_cols = [
        c for c in df.columns
        if c.startswith("cnt_") or c.startswith("tr_")
    ]

    if count_cols:
        df[count_cols] = df[count_cols].fillna(0)

    # Признаки отдельных типов действий
    contact_cols = [
        c for c in df.columns
        if c.startswith("cnt_contact")
    ]

    if contact_cols:
        df["contact_total"] = df[contact_cols].sum(axis=1)
        df["contact_rate"] = (
            df["contact_total"]
            / (df["event_ts_count"] + 1.0)
        )

    if "cnt_favorite_add" in df.columns:
        df["favorite_rate"] = (
            df["cnt_favorite_add"]
            / (df["event_ts_count"] + 1.0)
        )

    if "cnt_captcha_shown" in df.columns:
        df["has_captcha"] = (
            df["cnt_captcha_shown"] > 0
        ).astype("int8")

    if "cnt_login" in df.columns:
        df["has_login"] = (
            df["cnt_login"] > 0
        ).astype("int8")

    if {"cnt_contact_phone_show", "cnt_item_view"}.issubset(df.columns):
        df["phone_reveal_rate"] = (
            df["cnt_contact_phone_show"]
            / (df["cnt_item_view"] + 1.0)
        )

    if {"cnt_photo_swipe", "cnt_item_view"}.issubset(df.columns):
        df["photo_per_view"] = (
            df["cnt_photo_swipe"]
            / (df["cnt_item_view"] + 1.0)
        )

    return df.drop(
        columns=["event_ts_min", "event_ts_max"],
        errors="ignore"
    )


# Получаем итоговые признаки для train и test
X_train_all = assemble_features(train_meta)
X_test_all = assemble_features(test_meta)


feature_cols_all = [
    c for c in X_train_all.columns
    if c not in NON_FEATURE_COLS
]


# Базовый набор признаков по времени и pointer
KIN_PREFIXES = (
    "diff_ms", "is_fast", "is_pause", "is_burst",
    "pointer_", "same_eid", "same_event",
    "event_run", "eid_run", "burst_",
    "session", "kin_", "accel_"
)

# Более широкий набор поведенческих и relational-признаков
SEMANTIC_HINTS = (
    "cnt_", "tr_", "contact", "favorite", "captcha",
    "login", "platform", "reach", "prior", "gap",
    "first_seen", "item_", "query_", "ua_", "hour",
    "night", "weekend", "active_hours", "active_days",
    "switch", "novelty", "seller", "category",
    "location", "hist_", "cross_cookie", "session",
    "conc_", "first10", "first25", "last10", "last25"
)


# Kinematic-набор
feature_cols_kin = [
    c for c in feature_cols_all
    if any(c.startswith(p) for p in KIN_PREFIXES)
    or c in {
        "has_events", "cookie_age_hours",
        "window_duration_hours", "active_duration_sec",
        "active_window_coverage", "first_event_offset_sec",
        "last_event_offset_sec", "event_ts_count",
        "events_per_active_sec", "events_per_window_hour",
        "item_id_nunique", "item_category_nunique",
        "search_query_nunique", "user_agent_nunique",
    }
]


# Основные временные признаки для extended-набора
CORE_TIMING = {
    "diff_ms_mean", "diff_ms_std", "diff_ms_median",
    "diff_ms_min", "diff_ms_max", "diff_ms_q10",
    "diff_ms_q25", "diff_ms_q75", "diff_ms_q90",
    "diff_ms_q95", "diff_ms_iqr", "diff_ms_skew",
    "diff_ms_safe_entropy",
    "is_diff_zero_mean", "is_diff_zero_sum",
    "is_fast_100ms_mean", "is_fast_100ms_sum",
    "is_fast_500ms_mean", "is_fast_500ms_sum",
    "same_second_event_count_mean",
    "same_second_event_count_max",
    "same_second_event_count_std",
}


# Extended-набор
feature_cols_ext = [
    c for c in feature_cols_all
    if c in CORE_TIMING
    or any(c.startswith(p) for p in SEMANTIC_HINTS)
    or c in {
        "has_events", "cookie_age_hours",
        "window_duration_hours", "active_duration_sec",
        "active_window_coverage", "first_event_offset_sec",
        "last_event_offset_sec", "event_ts_count",
        "events_per_active_sec", "events_per_window_hour",
        "item_to_event_ratio", "category_to_event_ratio",
        "query_to_event_ratio", "ua_to_event_ratio",
        "pointer_event_ratio", "ua_rotation",
        "dominant_platform", "hour_sin_mean",
        "hour_cos_mean", "night_share",
        "event_hour_entropy", "active_hours_nunique",
        "active_days_nunique", "platform_switches",
        "ua_switches", "has_pointer_mean",
        "has_pointer_sum", "pointer_dist_mean",
        "pointer_dist_std", "pointer_dist_max",
        "pointer_zero_move_mean", "pointer_zero_move_sum",
        "ua_len_mean", "ua_len_min", "ua_len_max",
        "ua_len_std", "ua_is_bot_keyword_mean",
        "ua_is_bot_keyword_sum", "ua_is_bot_keyword_max",
        "ua_is_app_mean", "ua_is_app_sum",
        "search_page_max", "search_page_mean",
        "search_page_std", "search_page_diff_mean",
        "search_page_diff_max", "search_page_diff_std",
        "search_page_back_mean", "search_page_back_sum",
        "search_page_forward_mean",
        "search_page_forward_sum", "diff_ms_cv",
    }
]

feature_cols_ext = list(dict.fromkeys(feature_cols_ext))

# Hybrid объединяет оба набора
feature_cols_hybrid = list(
    dict.fromkeys(feature_cols_ext + feature_cols_kin)
)


# Убираем признаки без вариации
def remove_degenerate(frame: pd.DataFrame, cols: List[str]) -> List[str]:
    keep = []
    for c in cols:
        if c not in frame.columns:
            continue

        nun = frame[c].nunique(dropna=True)
        if nun > 1:
            keep.append(c)

    return keep


feature_cols_kin = remove_degenerate(
    X_train_all, feature_cols_kin
)

feature_cols_ext = remove_degenerate(
    X_train_all, feature_cols_ext
)

feature_cols_hybrid = remove_degenerate(
    X_train_all, feature_cols_hybrid
)


# CatBoost использует только эту категориальную переменную
CAT_FEATURES = (
    ["dominant_platform"]
    if "dominant_platform" in feature_cols_ext
    else []
)


print("\nFeature counts:")
print(f"  all      : {len(feature_cols_all)}")
print(f"  kinematic: {len(feature_cols_kin)}")
print(f"  extended : {len(feature_cols_ext)}")
print(f"  hybrid   : {len(feature_cols_hybrid)}")
print(
    f"  event types: "
    f"{len([c for c in X_train_all.columns if c.startswith('cnt_')])}"
)
print(
    f"  transitions: "
    f"{len([c for c in X_train_all.columns if c.startswith('tr_')])}"
)


Feature counts:
  all      : 341
  kinematic: 89
  extended : 241
  hybrid   : 289
  event types: 9
  transitions: 82


## 9. Temporal validation

Так как тестовая выборка относится к более позднему времени, случайное разбиение данных здесь было бы не совсем корректным. Поэтому я разделил train по календарным дням и стараюсь при проверке всегда предсказывать будущее по прошлому.

Вся обучающая выборка делится на три части:

calibration — самая ранняя часть данных. Она используется для walk-forward проверки и подбора весов итогового ансамбля;
selection — несколько следующих дней. На них подбираются гиперпараметры моделей;
audit — последние 7 дней train. Этот блок полностью откладывается до конца и не используется при настройке модели.

Для audit модель обучается на всех предыдущих днях и только после этого проверяется на последних днях. Таким образом, audit примерно имитирует ситуацию с настоящим скрытым тестом, где тестовые данные находятся в будущем.

Такое разделение позволяет отдельно проверить, насколько хорошо модель переносится на более поздний период, и уменьшить риск того, что результат получен из-за подгонки под конкретную валидационную выборку.

In [14]:
train_days = pd.to_datetime(train_meta["window_start_ts"]).dt.floor("D")
test_days = pd.to_datetime(test_meta["window_start_ts"]).dt.floor("D")
unique_train_days = np.array(sorted(train_days.unique()))
unique_test_days = np.array(sorted(test_days.unique()))


horizon_days = max(1, len(unique_test_days))

AUDIT_DAYS = min(horizon_days, max(3, horizon_days))
SELECTION_DAYS = min(2, max(1, horizon_days))

if len(unique_train_days) <= AUDIT_DAYS + SELECTION_DAYS + 4:
    n = len(unique_train_days)
    audit_n = max(2, int(round(n * 0.20)))
    select_n = max(2, int(round(n * 0.20)))
    AUDIT_DAYS, SELECTION_DAYS = audit_n, select_n

AUDIT_DAY_SET = set(unique_train_days[-AUDIT_DAYS:])
SELECT_DAY_SET = set(unique_train_days[-AUDIT_DAYS - SELECTION_DAYS:-AUDIT_DAYS])
CALIB_DAY_SET = set(unique_train_days[:-AUDIT_DAYS - SELECTION_DAYS])

if len(CALIB_DAY_SET) < 4 and len(unique_train_days) >= 10:
    SELECTION_DAYS = 1
    SELECT_DAY_SET = set(unique_train_days[-AUDIT_DAYS - SELECTION_DAYS:-AUDIT_DAYS])
    CALIB_DAY_SET = set(unique_train_days[:-AUDIT_DAYS - SELECTION_DAYS])

is_audit = train_days.isin(AUDIT_DAY_SET).to_numpy()
is_selection = train_days.isin(SELECT_DAY_SET).to_numpy()
is_calib = train_days.isin(CALIB_DAY_SET).to_numpy()


print(f"Train unique days: {len(unique_train_days)}")
print(f"Calibration days : {len(CALIB_DAY_SET)}")
print(f"Selection days   : {len(SELECT_DAY_SET)}")
print(f"Audit days      : {len(AUDIT_DAY_SET)}")
print(f"Calibration rows={is_calib.sum()}, positives={int(y_train[is_calib].sum())}")
print(f"Selection rows  ={is_selection.sum()}, positives={int(y_train[is_selection].sum())}")
print(f"Audit rows      ={is_audit.sum()}, positives={int(y_train[is_audit].sum())}")
print(f"Audit: {min(AUDIT_DAY_SET)} -> {max(AUDIT_DAY_SET)}")


Train unique days: 14
Calibration days : 5
Selection days   : 2
Audit days      : 7
Calibration rows=4217, positives=345
Selection rows  =1713, positives=146
Audit rows      =5161, positives=408
Audit: 2026-04-13 00:00:00 -> 2026-04-19 00:00:00


Здесь я добавила признаки, которые показывают, насколько часто в прошлом определённый объект или признак был связан с ботами. Для этого использовались `item_id`, `search_query`, `user_agent`, категория товара и локация.

Сначала для каждого такого признака собиралась статистика только по уже размеченным `train` данным. Для каждого значения считалось количество cookie, которые с ним встречались, и количество из них, которые были ботами. После этого вычислялась сглаженная доля ботов. Сглаживание нужно, чтобы редкие значения с одной-двумя встречами не давали слишком экстремальные значения.

При расчёте для конкретного cookie я использовал только информацию из прошлого. Для `train` текущий день не включался в историю, поэтому модель не могла напрямую получить информацию о метке из того же дня. Для теста использовались все доступные размеченные дни из `train`.

После этого из исторической статистики формировались несколько признаков: средняя историческая доля ботов по связанным объектам, максимальная доля, высокий квантиль, а также количество значений, для которых вообще была найдена история. Если текущий `item_id`, запрос или `User-Agent` раньше часто встречались у ботов, это могло дать модели дополнительный сигнал.

Также при подсчёте учитывалось, появлялся ли этот же cookie раньше. Его собственный вклад в историческую статистику исключался, чтобы не использовать информацию из самого объекта при построении признака.

В итоге этот блок добавляет не просто информацию о текущем поведении пользователя, а ещё и небольшой контекст о том, как похожие объекты и признаки вели себя раньше.


In [15]:
def _history_table_for_key(events_df: pd.DataFrame, train_meta_df: pd.DataFrame, y_arr: np.ndarray, key_col: str):
    # Берём train cookie
    train_ids = set(train_meta_df["cookie_id"].astype("string"))

    # Оставляем нужный ключ
    src = events_df[events_df["cookie_id"].astype("string").isin(train_ids)][
        ["cookie_id", key_col]
    ].copy()

    # Убираем пропуски
    src = src[src[key_col].notna()]

    if src.empty:
        return None

    # Сохраняем метки
    label_map = pd.Series(y_arr.astype(int), index=train_meta_df["cookie_id"].astype("string")).to_dict()

    # Определяем день
    day_map = pd.Series(
        pd.to_datetime(train_meta_df["window_start_ts"]).dt.floor("D").to_numpy(),
        index=train_meta_df["cookie_id"].astype("string"),
    ).to_dict()

    src["_cid"] = src["cookie_id"].astype("string")
    src["_day"] = src["_cid"].map(day_map)
    src["_target"] = src["_cid"].map(label_map).astype(int)

    # Убираем повторы
    src = src.drop_duplicates([key_col, "_cid"])

    # Считаем cookie и ботов
    daily = src.groupby([key_col, "_day"], observed=True).agg(
        n_cookies=("_cid", "nunique"),
        n_bots=("_target", "sum"),
    ).reset_index()

    daily = daily.sort_values([key_col, "_day"], kind="mergesort")

    # Накопленная история
    daily["cum_cookies_total"] = daily.groupby(key_col, observed=True)["n_cookies"].cumsum()
    daily["cum_bots_total"] = daily.groupby(key_col, observed=True)["n_bots"].cumsum()

    # Первое появление
    first_pair = src.groupby([key_col, "_cid"], observed=True)["_day"].min().reset_index(name="first_day")

    return daily, first_pair


def add_historical_bot_rate_features(X_train: pd.DataFrame, X_test: pd.DataFrame):
    # Определяем дни
    train_days_local = pd.to_datetime(train_meta["window_start_ts"]).dt.floor("D")
    test_days_local = pd.to_datetime(test_meta["window_start_ts"]).dt.floor("D")

    # Настраиваем cutoff
    train_cutoff = train_days_local - pd.Timedelta(days=0)
    test_cutoff = pd.Series(max(train_days_local) + pd.Timedelta(days=1), index=test_meta.index)

    train_cutoff_map = pd.Series(train_cutoff.to_numpy(), index=train_meta["cookie_id"].astype("string")).to_dict()
    test_cutoff_map = pd.Series(test_cutoff.to_numpy(), index=test_meta["cookie_id"].astype("string")).to_dict()

    # Берём события
    ev = events[["cookie_id", "event_ts"] + [c for c in HISTORY_KEYS if c in events.columns]].copy()
    ev["_cid"] = ev["cookie_id"].astype("string")

    # Отмечаем test
    ev["_is_test"] = ev["_cid"].isin(set(test_meta["cookie_id"].astype("string"))).astype("int8")

    # Добавляем cutoff
    ev["_cutoff"] = ev["_cid"].map(train_cutoff_map)
    ev.loc[ev["_is_test"].eq(1), "_cutoff"] = ev.loc[ev["_is_test"].eq(1), "_cid"].map(test_cutoff_map)
    ev["_cutoff"] = pd.to_datetime(ev["_cutoff"]).dt.floor("D")

    for key_col in HISTORY_KEYS:
        if key_col not in ev.columns:
            continue

        # Строим историю
        built = _history_table_for_key(events, train_meta, y_train, key_col)
        if built is None:
            continue

        daily, first_pair = built

        # Индексы истории
        daily_groups = daily.groupby(key_col, observed=True).indices
        pair_lookup = first_pair.set_index([key_col, "_cid"])["first_day"].to_dict()

        rate = np.full(len(ev), np.nan, dtype=np.float32)
        known = np.zeros(len(ev), dtype=np.float32)

        valid = ev[key_col].notna().to_numpy()
        valid_pos = np.flatnonzero(valid)
        temp = ev.loc[valid, [key_col, "_cid", "_cutoff"]].reset_index(drop=True)
        temp["_ev_pos"] = valid_pos

        grouped = temp.groupby(key_col, observed=True).groups

        for key_value, idx in grouped.items():
            if key_value not in daily_groups:
                continue

            # Берём прошлую историю
            ri = np.asarray(daily_groups[key_value], dtype=np.int64)
            dsub = daily.iloc[ri].sort_values("_day", kind="mergesort")
            days_i = dsub["_day"].astype("int64").to_numpy()
            cum_n = dsub["cum_cookies_total"].to_numpy(dtype=np.float64)
            cum_b = dsub["cum_bots_total"].to_numpy(dtype=np.float64)
            li_local = np.asarray(idx, dtype=np.int64)
            cutoff = temp.iloc[li_local]["_cutoff"].astype("int64").to_numpy()

            # Ищем историю
            pos = np.searchsorted(days_i, cutoff, side="left")
            n0 = np.zeros(len(pos), dtype=np.float64)
            b0 = np.zeros(len(pos), dtype=np.float64)
            nz = pos > 0
            n0[nz] = cum_n[pos[nz] - 1]
            b0[nz] = cum_b[pos[nz] - 1]

            # Убираем свой вклад
            cids = temp.iloc[li_local]["_cid"].astype("string").to_numpy()
            for j, cid in enumerate(cids):
                fd = pair_lookup.get((key_value, cid))
                if fd is not None and int(fd.value) < int(cutoff[j]):
                    n0[j] = max(0.0, n0[j] - 1.0)
                    y_self = label_map_global.get(cid)
                    if y_self == 1:
                        b0[j] = max(0.0, b0[j] - 1.0)

            # Считаем bot-rate
            rates = (b0 + HISTORY_ALPHA) / (n0 + 2.0 * HISTORY_ALPHA)
            dest = temp.iloc[li_local]["_ev_pos"].to_numpy(dtype=np.int64)
            rate[dest] = rates.astype(np.float32)
            known[dest] = n0.astype(np.float32)

        # Добавляем признаки
        prefix = f"hist_{key_col}"
        ev[f"{prefix}_botrate"] = rate
        ev[f"{prefix}_known"] = known

        # Собираем по cookie
        h = ev.groupby(ev["_cid"], observed=True).agg(
            botrate_mean=(f"{prefix}_botrate", "mean"),
            botrate_max=(f"{prefix}_botrate", "max"),
            botrate_q90=(f"{prefix}_botrate", q90),
            known_mean=(f"{prefix}_known", "mean"),
            known_max=(f"{prefix}_known", "max"),
            known_share=(f"{prefix}_known", lambda s: float((s > 0).mean())),
        ).reset_index().rename(columns={"_cid": "cookie_id"})

        h.columns = ["cookie_id" if c == "cookie_id" else f"{prefix}_{c}" for c in h.columns]

        # Добавляем в train/test
        X_train = X_train.merge(h, on="cookie_id", how="left")
        X_test = X_test.merge(h, on="cookie_id", how="left")

    return X_train, X_test


# Сохраняем метки
label_map_global = pd.Series(y_train.astype(int), index=train_meta["cookie_id"].astype("string")).to_dict()

# Добавляем историю
X_train_all, X_test_all = add_historical_bot_rate_features(X_train_all, X_test_all)

# Собираем новые признаки
history_cols = [c for c in X_train_all.columns if c.startswith("hist_")]

feature_cols_all = list(dict.fromkeys(feature_cols_all + history_cols))
feature_cols_ext = list(dict.fromkeys(feature_cols_ext + history_cols))

feature_cols_hybrid = list(dict.fromkeys(feature_cols_hybrid + history_cols + [
    c for c in ["largest_session_fraction", "sessions_per_hour"] if c in X_train_all.columns
]))

feature_cols_ext = list(dict.fromkeys(feature_cols_ext + [
    c for c in ["largest_session_fraction", "sessions_per_hour"] if c in X_train_all.columns
]))

feature_cols_all = list(dict.fromkeys(feature_cols_all + [
    c for c in ["largest_session_fraction", "sessions_per_hour"] if c in X_train_all.columns
]))

# Убираем лишние признаки
feature_cols_kin = remove_degenerate(X_train_all, feature_cols_kin)
feature_cols_ext = remove_degenerate(X_train_all, feature_cols_ext)
feature_cols_hybrid = remove_degenerate(X_train_all, feature_cols_hybrid)

## 10. Фабрики моделей

В этом блоке я сделала отдельные функции для создания трёх моделей: LightGBM, CatBoost и XGBoost. Это позволяет не писать настройки моделей каждый раз заново и просто передавать нужные параметры и seed.

Также здесь один раз рассчитывается class weight, потому что в данных ботов заметно меньше, чем обычных пользователей. Поэтому при обучении модели больший вес получают объекты класса bot, чтобы модель не могла просто часто предсказывать основной класс.

Для всех трёх моделей используются примерно одинаковые общие настройки: фиксируется seed, задаётся количество деревьев, глубина или сложность модели и параметры регуляризации. При этом у каждой библиотеки остаются свои собственные параметры.

Дальше эти функции используются при подборе гиперпараметров и при финальном обучении. Благодаря этому можно легко обучить несколько вариантов одной и той же модели с разными параметрами и случайными сидами.

In [16]:
# Считаем вес класса
def pos_weight(y):
    y = np.asarray(y)
    p = float(y.sum())
    n = float(len(y) - p)
    return n / max(p, 1.0)

# Создаём LightGBM
def make_lgb(params: Dict, n_estimators: int, y: np.ndarray, seed: Optional[int] = None):
    p = dict(params)
    if "scale_pos_weight" not in p or p["scale_pos_weight"] is None:
        p["scale_pos_weight"] = pos_weight(y)
    if seed is not None:
        p["random_state"] = seed
    return lgb.LGBMClassifier(**p, n_estimators=n_estimators)

# категориальные признаки
def prepare_cat(df: pd.DataFrame, cols: List[str]) -> pd.DataFrame:
    out = df[cols].copy()
    if "dominant_platform" in out.columns:
        out["dominant_platform"] = out["dominant_platform"].astype(str).fillna("missing")
    return out

# Создаём CatBoost
def make_cat(params: Dict, iterations: int, y: np.ndarray, catcols: List[str], seed: int = SEED):
    p = dict(params)
    p["scale_pos_weight"] = pos_weight(y)
    return CatBoostClassifier(
        **p,
        iterations=iterations,
        cat_features=catcols,
        verbose=False,
        random_seed=seed,
    )

# Создаём XGBoost
def make_xgb(params: Dict, n_estimators: int, y: np.ndarray, seed: Optional[int] = None):
    p = dict(params)
    if "scale_pos_weight" not in p or p["scale_pos_weight"] is None:
        p["scale_pos_weight"] = pos_weight(y)
    if seed is not None:
        p["random_state"] = seed
    return xgb.XGBClassifier(
        **p,
        n_estimators=n_estimators,
        tree_method="hist",
    )

## 11. Baseline

Сначала я сделал простой baseline на основе LightGBM. В него вошёл только базовый набор признаков, без более сложных исторических и поведенческих признаков, которые добавляются дальше.

Модель обучается на более ранней части `train`, а затем проверяется на `selection block`, то есть на следующих по времени днях. Для оценки используется основная метрика соревнования — `Precision@Recall >= 0.70`.

Этот baseline нужен как отправная точка. Сначала я смотрю, какой результат можно получить с достаточно простой моделью, а затем сравниваю его с расширенными вариантами. Таким образом, можно понять, действительно ли дополнительные признаки и более сложный ансамбль дают прирост, а не просто усложняют решение.


In [17]:
# Базовые параметры
BASE_PARAMS = {
    "objective": "binary",
    "metric": "average_precision",
    "learning_rate": 0.03,
    "num_leaves": 31,
    "max_depth": 5,
    "min_child_samples": 40,
    "feature_fraction": 0.80,
    "bagging_fraction": 0.80,
    "bagging_freq": 1,
    "random_state": SEED,
    "n_jobs": -1,
    "verbose": -1,
    "lambda_l1": 0.0,
    "lambda_l2": 3.0,
}

# Выбираем базовые признаки
BASE_COLS = remove_degenerate(
    X_train_all,
    [
        "has_events", "cookie_age_hours", "window_duration_hours", "active_duration_sec",
        "active_window_coverage", "event_ts_count", "events_per_active_sec",
        "item_id_nunique", "item_category_nunique", "item_location_nunique",
        "seller_type_nunique", "search_query_nunique", "user_agent_nunique",
        "diff_ms_mean", "diff_ms_std", "diff_ms_median", "diff_ms_min", "diff_ms_max",
        "pointer_dist_mean", "pointer_dist_std", "pointer_dist_max",
        "pointer_speed_mean", "pointer_speed_std", "pointer_speed_max",
        "has_pointer_mean", "has_pointer_sum", "pointer_zero_move_mean", "pointer_zero_move_sum",
        "ua_len_mean", "ua_len_min", "ua_len_max", "platform_nunique", "event_name_nunique",
    ],
)

base_model = make_lgb(BASE_PARAMS, 2500, y_train[~is_selection & ~is_audit])

Xb_tr = X_train_all.loc[~is_selection & ~is_audit, BASE_COLS]
Xb_va = X_train_all.loc[is_selection, BASE_COLS]

yb_tr = y_train[~is_selection & ~is_audit]
yb_va = y_train[is_selection]

# Обучаем модель
base_model.fit(
    Xb_tr, yb_tr,
    eval_set=[(Xb_va, yb_va)],
    callbacks=[
        lgb.early_stopping(EARLY_STOPPING, verbose=False),
        lgb.log_evaluation(0)
    ],
)


p_base_sel = base_model.predict_proba(Xb_va)[:, 1]

base_selection_score = score_pr(yb_va, p_base_sel)

print(
    f"Selection baseline P@R0.70 = {base_selection_score:.4f}; "
    f"best_iter={base_model.best_iteration_}"
)

Selection baseline P@R0.70 = 0.7123; best_iter=70


## 12. Подбор гиперпараметров

После baseline я отдельно подбирала параметры для LightGBM, CatBoost и XGBoost. Для каждой модели задавался небольшой набор вариантов параметров, например глубина деревьев, количество листьев, learning rate и другие параметры, которые влияют на сложность модели.

Все эти эксперименты проводились только на `selection block`. Для каждого варианта модель обучалась на более ранних данных и проверялась на `selection`, после чего сравнивался `Precision@Recall >= 0.70`. Таким образом, параметры выбирались по результату на отложенном временном блоке, а не на тех же данных, на которых модель обучалась.

После выбора лучших вариантов найденные параметры фиксируются. Последние 7 дней `audit` при этом вообще не используются для подбора. Они нужны только для финальной проверки того, как выбранная модель работает на более позднем периоде.


In [18]:
rng = np.random.default_rng(SEED)

# сэмплирование гиперпараметров для LightGBM
def sampled_lgb_params():
    return {
        "objective": "binary",
        "metric": "average_precision",
        "learning_rate": float(rng.choice([0.015, 0.02, 0.03, 0.05])),
        "num_leaves": int(rng.choice([15, 24, 31, 47, 63])),
        "max_depth": int(rng.choice([4, 5, 6, 7, -1])),
        "min_child_samples": int(rng.choice([15, 25, 40, 60, 90])),
        "feature_fraction": float(rng.choice([0.65, 0.75, 0.85, 1.0])),
        "bagging_fraction": float(rng.choice([0.75, 0.85, 1.0])),
        "bagging_freq": 1,
        "lambda_l1": float(rng.choice([0.0, 0.1, 1.0, 5.0])),
        "lambda_l2": float(rng.choice([1.0, 3.0, 7.0, 15.0])),
        "scale_pos_weight": float(rng.choice([1.0, 1.5, 2.5, 4.0, 7.0, 10.0])),
        "random_state": SEED,
        "n_jobs": -1,
        "verbose": -1,
    }

# тюнинг LightGBM через Random Search
def tune_lgb(frame, cols, label):
    Xtr = frame.loc[~is_selection & ~is_audit, cols]
    Xsel = frame.loc[is_selection, cols]
    ytr = y_train[~is_selection & ~is_audit]
    ysel = y_train[is_selection]
    records = []
    for t in range(N_LGB_TRIALS):
        p = sampled_lgb_params()
        m = make_lgb(p, 3000, ytr)
        m.fit(
            Xtr, ytr,
            eval_set=[(Xsel, ysel)],
            callbacks=[lgb.early_stopping(EARLY_STOPPING, verbose=False), lgb.log_evaluation(0)],
        )
        pred = m.predict_proba(Xsel)[:, 1]
        sc = score_pr(ysel, pred)
        records.append({**p, "best_iteration": int(m.best_iteration_ or 800), "score": sc})
        print(f"  {label} LGB trial {t+1:02d}/{N_LGB_TRIALS}: {sc:.4f}")
    df = pd.DataFrame(records).sort_values("score", ascending=False).reset_index(drop=True)
    best = df.iloc[0].to_dict()
    it = int(best.pop("best_iteration"))
    best.pop("score")
    for c in ["num_leaves", "max_depth", "min_child_samples"]:
        best[c] = int(best[c])
    return best, it, df

# тюнинг CatBoost
def tune_cat(frame, cols, label):
    Xtr = prepare_cat(frame.loc[~is_selection & ~is_audit], cols)
    Xsel = prepare_cat(frame.loc[is_selection], cols)
    ytr = y_train[~is_selection & ~is_audit]
    ysel = y_train[is_selection]
    catcols = [c for c in ["dominant_platform"] if c in cols]
    records = []
    for t in range(N_CB_TRIALS):
        p = {
            "loss_function": "Logloss",
            "eval_metric": "AUC",
            "learning_rate": float(rng.choice([0.015, 0.02, 0.03, 0.05])),
            "depth": int(rng.choice([4, 5, 6, 7, 8])),
            "l2_leaf_reg": float(rng.choice([1.0, 3.0, 5.0, 10.0, 20.0])),
            "random_strength": float(rng.choice([0.0, 0.5, 1.0, 2.0])),
        }
        m = make_cat(p, 2500, ytr, catcols)
        m.fit(
            Xtr, ytr,
            eval_set=(Xsel, ysel),
            early_stopping_rounds=EARLY_STOPPING,
            use_best_model=True,
        )
        pred = m.predict_proba(Xsel)[:, 1]
        sc = score_pr(ysel, pred)
        records.append({**p, "iterations": int(m.get_best_iteration() + 1), "score": sc})
        print(f"  {label} CAT trial {t+1:02d}/{N_CB_TRIALS}: {sc:.4f}")
    df = pd.DataFrame(records).sort_values("score", ascending=False).reset_index(drop=True)
    best = df.iloc[0].to_dict()
    it = int(best.pop("iterations"))
    best.pop("score")
    best["depth"] = int(best["depth"])
    return best, it, df

# тюнинг XGBoost
def tune_xgb(frame, cols, label):
    Xtr = frame.loc[~is_selection & ~is_audit, cols].copy()
    Xsel = frame.loc[is_selection, cols].copy()
    for c in ["dominant_platform"]:
        if c in Xtr.columns:
            cats = pd.concat([Xtr[c].astype("string"), Xsel[c].astype("string")]).fillna("missing").astype("category")
            Xtr[c] = cats.iloc[:len(Xtr)].cat.codes.astype("int16")
            Xsel[c] = cats.iloc[len(Xtr):].cat.codes.astype("int16")
    ytr = y_train[~is_selection & ~is_audit]
    ysel = y_train[is_selection]
    records = []
    for t in range(N_XGB_TRIALS):
        p = {
            "objective": "binary:logistic",
            "eval_metric": "aucpr",
            "learning_rate": float(rng.choice([0.02, 0.03, 0.05])),
            "max_depth": int(rng.choice([3, 4, 5, 6])),
            "min_child_weight": int(rng.choice([1, 3, 5, 10])),
            "subsample": float(rng.choice([0.75, 0.85, 1.0])),
            "colsample_bytree": float(rng.choice([0.65, 0.8, 1.0])),
            "reg_lambda": float(rng.choice([1.0, 5.0, 10.0])),
            "scale_pos_weight": float(rng.choice([1.0, 1.5, 2.5, 4.0, 7.0])),
            "random_state": SEED,
            "n_jobs": -1,
        }
        m = make_xgb(p, 2200, ytr)
        m.fit(
            Xtr, ytr,
            eval_set=[(Xsel, ysel)],
            verbose=False,
        )
        pred = m.predict_proba(Xsel)[:, 1]
        sc = score_pr(ysel, pred)
        records.append({**p, "n_estimators": 2200, "score": sc})
        print(f"  {label} XGB trial {t+1:02d}/{N_XGB_TRIALS}: {sc:.4f}")
    df = pd.DataFrame(records).sort_values("score", ascending=False).reset_index(drop=True)
    best = df.iloc[0].to_dict()
    best.pop("score")
    best.pop("n_estimators")
    for c in ["max_depth", "min_child_weight"]:
        best[c] = int(best[c])
    return best, 2200, df

# гиперпараметры для всех моделей
best_ext_lgb_params, best_ext_lgb_n, ext_lgb_trials = tune_lgb(
    X_train_all, feature_cols_ext, "extended"
)
best_ext_cat_params, best_ext_cat_n, ext_cat_trials = tune_cat(
    X_train_all, feature_cols_ext, "extended"
)
best_hybrid_lgb_params, best_hybrid_lgb_n, hybrid_lgb_trials = tune_lgb(
    X_train_all, feature_cols_hybrid, "hybrid"
)

best_ext_xgb_params, best_ext_xgb_n, ext_xgb_trials = tune_xgb(
    X_train_all, feature_cols_ext, "extended"
)

  extended LGB trial 01/6: 0.8443
  extended LGB trial 02/6: 0.8047
  extended LGB trial 03/6: 0.7591
  extended LGB trial 04/6: 0.7591
  extended LGB trial 05/6: 0.7518
  extended LGB trial 06/6: 0.7863
  extended CAT trial 01/6: 0.6980
  extended CAT trial 02/6: 0.7153
  extended CAT trial 03/6: 0.6519
  extended CAT trial 04/6: 0.6753
  extended CAT trial 05/6: 0.6887
  extended CAT trial 06/6: 0.6341
  hybrid LGB trial 01/6: 0.8443
  hybrid LGB trial 02/6: 0.7761
  hybrid LGB trial 03/6: 0.7803
  hybrid LGB trial 04/6: 0.7376
  hybrid LGB trial 05/6: 0.7630
  hybrid LGB trial 06/6: 0.7744
  extended XGB trial 01/2: 0.7153
  extended XGB trial 02/2: 0.7410


## 13. Walk-forward OOF

Здесь я делаю временную кросс-валидацию, чтобы проверить модели в условиях, которые больше похожи на реальный тест. Данные не перемешиваются: для каждого шага модель обучается только на днях, которые уже прошли, и затем делает предсказания для следующего временного блока.

Например, если на текущем шаге доступны первые несколько дней, модель обучается на них и предсказывает следующий день. Потом этот следующий день добавляется в обучающую историю, и процедура повторяется для следующего шага. Поэтому в каждом OOF-предсказании модель не видит будущее.

Полученные предсказания собираются вместе в один OOF-массив. На них я отдельно считаю `Precision@Recall >= 0.70` для разных моделей и смотрю, какие эксперты дают полезный результат. Эти OOF-предсказания потом используются ещё и для подбора весов итогового ансамбля.

Такой подход удобнее обычной случайной кросс-валидации, потому что сохраняет временной порядок данных и позволяет лучше оценить, как модель будет работать на более поздних периодах.


In [19]:
# список моделей и массив под OOF-предсказания
EXPERTS = ["ext_lgb", "ext_cat", "hybrid_lgb", "ext_xgb"]
wf_oof = {k: np.full(len(train_meta), np.nan, dtype=float) for k in EXPERTS}

# дни для калибровки и период прогрева
CALIB_FOR_WF = np.array(sorted(set(unique_train_days) - AUDIT_DAY_SET))
WF_BURN_IN = min(7, max(3, len(CALIB_FOR_WF) // 3))

# функция обучения одной модели
def fit_expert_predict(expert: str, train_mask: np.ndarray, valid_mask: np.ndarray):
    ytr = y_train[train_mask]
    if expert == "ext_lgb":
        m = make_lgb(best_ext_lgb_params, best_ext_lgb_n, ytr)
        m.fit(X_train_all.loc[train_mask, feature_cols_ext], ytr)
        return m.predict_proba(X_train_all.loc[valid_mask, feature_cols_ext])[:, 1]

    if expert == "ext_cat":
        tr = prepare_cat(X_train_all.loc[train_mask], feature_cols_ext)
        va = prepare_cat(X_train_all.loc[valid_mask], feature_cols_ext)
        m = make_cat(best_ext_cat_params, best_ext_cat_n, ytr, CAT_FEATURES)
        m.fit(tr, ytr)
        return m.predict_proba(va)[:, 1]

    if expert == "hybrid_lgb":
        m = make_lgb(best_hybrid_lgb_params, best_hybrid_lgb_n, ytr)
        m.fit(X_train_all.loc[train_mask, feature_cols_hybrid], ytr)
        return m.predict_proba(X_train_all.loc[valid_mask, feature_cols_hybrid])[:, 1]

    if expert == "ext_xgb":
        tr = X_train_all.loc[train_mask, feature_cols_ext].copy()
        va = X_train_all.loc[valid_mask, feature_cols_ext].copy()
        for c in ["dominant_platform"]:
            if c in tr.columns:
                cats = pd.concat([tr[c].astype("string"), va[c].astype("string")]).fillna("missing").astype("category")
                tr[c] = cats.iloc[:len(tr)].cat.codes.astype("int16")
                va[c] = cats.iloc[len(tr):].cat.codes.astype("int16")
        m = make_xgb(best_ext_xgb_params, best_ext_xgb_n, ytr)
        m.fit(tr, ytr, verbose=False)
        return m.predict_proba(va)[:, 1]

    raise KeyError(expert)

# циклами проходим по дням
for i in range(WF_BURN_IN, len(CALIB_FOR_WF)):
    day = CALIB_FOR_WF[i]
    valid_mask = train_days.eq(day).to_numpy()
    train_mask = train_days.lt(day).to_numpy()
    if valid_mask.sum() == 0 or y_train[valid_mask].sum() < 5:
        continue
    print(f"WF {pd.Timestamp(day).date()}: n={valid_mask.sum()}, pos={int(y_train[valid_mask].sum())}")
    for expert in EXPERTS:
        wf_oof[expert][valid_mask] = fit_expert_predict(expert, train_mask, valid_mask)

# собираем маску валидных предсказаний
wf_mask = np.ones(len(train_meta), dtype=bool)
for expert in EXPERTS:
    wf_mask &= np.isfinite(wf_oof[expert])
wf_mask &= is_calib

# проверка, что данных хватит
if wf_mask.sum() < 500 or y_train[wf_mask].sum() < 30:
    raise RuntimeError(
        "Слишком мало pooled WF OOF для устойчивого blend search. "
        "Проверьте количество train-дней или уменьшите AUDIT_DAYS/SELECTION_DAYS."
    )

# считаем и выводим метрики для каждой модели
print(f"WF rows={wf_mask.sum()}, positives={int(y_train[wf_mask].sum())}")
for expert in EXPERTS:
    print(f"  {expert:12s}: {score_pr(y_train[wf_mask], wf_oof[expert][wf_mask]):.4f}")

WF 2026-04-09: n=902, pos=66
WF 2026-04-10: n=912, pos=82
WF 2026-04-11: n=896, pos=79
WF 2026-04-12: n=817, pos=67
WF rows=1814, positives=148
  ext_lgb     : 0.8125
  ext_cat     : 0.7075
  hybrid_lgb  : 0.7955
  ext_xgb     : 0.8231


## 14. Rank-space blend

После получения OOF-предсказаний я решил объединить несколько моделей в один итоговый прогноз. Здесь используется не прямое среднее вероятностей, а перевод предсказаний каждой модели в ранги. То есть объекты сортируются по предсказанной вероятности, и вместо самого значения используется его положение в этом порядке.

Это удобно в данном случае, потому что основная метрика зависит в первую очередь от порядка объектов. Поэтому не так важно, что одна модель выдаёт значения от 0 до 1, а другая, например, более сжатые вероятности. После перевода в ранги предсказания разных моделей становятся более сопоставимыми.

Дальше я перебираю разные веса для моделей и степень преобразования итогового ранга. Для каждого варианта считается итоговый score на calibration OOF. При этом selection и audit для этого подбора не используются.

В результате выбирается комбинация весов, которая лучше всего работает на этих OOF-предсказаниях. Затем те же зафиксированные веса используются уже для объединения предсказаний моделей на следующих этапах.

In [20]:
# нормировка предсказаний в ранги от 0 до 1
def rank01(a):
    return rankdata(np.asarray(a), method="average") / len(a)

R = {e: rank01(wf_oof[e][wf_mask]) for e in EXPERTS}
y_wf = y_train[wf_mask]

# функция оценки метрики для выбранных весов и степени
def eval_weights(weights, power):
    pred = np.zeros(len(y_wf), dtype=float)
    for e, w in zip(EXPERTS, weights):
        if w:
            pred += w * np.power(R[e], power)
    return score_pr(y_wf, pred)

best_score = -1.0
best_weights = None
best_power = 1.0

# грубый перебор весов и степеней по сетке
for pwr in POWER_GRID:
    step = BLEND_STEP_COARSE
    for a in np.arange(0, 1 + 1e-9, step):
        for b in np.arange(0, 1 - a + 1e-9, step):
            for c in np.arange(0, 1 - a - b + 1e-9, step):
                d = 1 - a - b - c
                weights = (a, b, c, d)
                sc = eval_weights(weights, pwr)
                if sc > best_score:
                    best_score, best_weights, best_power = sc, weights, pwr

# точечный добор весов в окрестности лучшей точки
center = np.array(best_weights, dtype=float)
fine_values = [
    np.clip(center[0] + z, 0, 1) for z in np.arange(-0.10, 0.1001, BLEND_STEP_FINE)
]
for a in fine_values:
    for b in [np.clip(center[1] + z, 0, 1) for z in np.arange(-0.10, 0.1001, BLEND_STEP_FINE)]:
        for c in [np.clip(center[2] + z, 0, 1) for z in np.arange(-0.10, 0.1001, BLEND_STEP_FINE)]:
            d = 1 - a - b - c
            if d < -1e-9:
                continue
            d = float(max(0.0, d))
            weights = (float(a), float(b), float(c), d)
            sc = eval_weights(weights, best_power)
            if sc > best_score:
                best_score, best_weights = sc, weights

we_lgb, we_cat, wh_lgb, we_xgb = best_weights

print(f"Weights: ext_lgb={we_lgb:.2f}, ext_cat={we_cat:.2f}, hybrid_lgb={wh_lgb:.2f}, ext_xgb={we_xgb:.2f}")
print(f"Power  : {best_power:.2f}")
print(f"Pooled WF blend={best_score:.4f}")

Weights: ext_lgb=0.50, ext_cat=0.00, hybrid_lgb=0.50, ext_xgb=0.00
Power  : 1.15
Pooled WF blend=0.8254


## 15. Финальный temporal audit

На этом этапе все решения, которые были приняты на предыдущих этапах, фиксируются, и после этого модель проверяется на последних 7 днях `train`, которые заранее были отложены как `audit`.

Модель обучается на всех предыдущих днях, то есть на данных до начала `audit`, и затем делает предсказания только для последних дней. Таким образом, эти данные имитируют будущий тест: во время обучения модель их вообще не видит.

На `audit` я считаю основную метрику `Precision@Recall >= 0.70` и отдельно смотрю результаты отдельных моделей и итогового blend. Это позволяет понять, сохраняется ли качество на самом позднем периоде, а не только на `selection`, который использовался при настройке.

Важно, что результаты `audit` не используются для изменения гиперпараметров, выбора признаков или весов ансамбля. Поэтому этот блок служит именно для финальной независимой проверки уже готового решения.


In [ ]:
# разделяем на обучающую выборку и аудит
pre_audit = ~is_audit
y_pre = y_train[pre_audit]

audit_preds = {}

# lgb
m_audit_ext_lgb = make_lgb(best_ext_lgb_params, best_ext_lgb_n, y_pre)
m_audit_ext_lgb.fit(X_train_all.loc[pre_audit, feature_cols_ext], y_pre)
audit_preds["ext_lgb"] = m_audit_ext_lgb.predict_proba(X_train_all.loc[is_audit, feature_cols_ext])[:, 1]

# catboost
audit_tr_cat = prepare_cat(X_train_all.loc[pre_audit], feature_cols_ext)
audit_va_cat = prepare_cat(X_train_all.loc[is_audit], feature_cols_ext)
m_audit_ext_cat = make_cat(best_ext_cat_params, best_ext_cat_n, y_pre, CAT_FEATURES)
m_audit_ext_cat.fit(audit_tr_cat, y_pre)
audit_preds["ext_cat"] = m_audit_ext_cat.predict_proba(audit_va_cat)[:, 1]

# hybrid lgb
m_audit_hyb = make_lgb(best_hybrid_lgb_params, best_hybrid_lgb_n, y_pre)
m_audit_hyb.fit(X_train_all.loc[pre_audit, feature_cols_hybrid], y_pre)
audit_preds["hybrid_lgb"] = m_audit_hyb.predict_proba(X_train_all.loc[is_audit, feature_cols_hybrid])[:, 1]

# xgb (кодируем категории)
audit_tr_xgb = X_train_all.loc[pre_audit, feature_cols_ext].copy()
audit_va_xgb = X_train_all.loc[is_audit, feature_cols_ext].copy()
if "dominant_platform" in audit_tr_xgb.columns:
    cats = pd.concat([
        audit_tr_xgb["dominant_platform"].astype("string"),
        audit_va_xgb["dominant_platform"].astype("string"),
    ]).fillna("missing").astype("category")
    audit_tr_xgb["dominant_platform"] = cats.iloc[:len(audit_tr_xgb)].cat.codes.astype("int16")
    audit_va_xgb["dominant_platform"] = cats.iloc[len(audit_tr_xgb):].cat.codes.astype("int16")

m_audit_xgb = make_xgb(best_ext_xgb_params, best_ext_xgb_n, y_pre)
m_audit_xgb.fit(audit_tr_xgb, y_pre, verbose=False)
audit_preds["ext_xgb"] = m_audit_xgb.predict_proba(audit_va_xgb)[:, 1]

y_audit = y_train[is_audit]

# скоры отдельных моделей
for name in EXPERTS:
    print(f"{name:12s}: {score_pr(y_audit, audit_preds[name]):.4f}")

# ансамбль (бленд)
R_audit = {e: rank01(audit_preds[e]) for e in EXPERTS}
p_audit_blend = np.zeros(len(y_audit), dtype=float)
for e, w in zip(EXPERTS, best_weights):
    p_audit_blend += w * np.power(R_audit[e], best_power)

# сравнение с лучшей моделью
audit_ensemble_score = score_pr(y_audit, p_audit_blend)
print(f"AUDIT BLEND   : {audit_ensemble_score:.4f}")
print(f"AUDIT BEST SINGLE: {max(score_pr(y_audit, audit_preds[e]) for e in EXPERTS):.4f}")
print(f"AUDIT gain over best single: {audit_ensemble_score - max(score_pr(y_audit, audit_preds[e]) for e in EXPERTS):+.4f}")

# доверительный интервал (бутстрэп)
def bootstrap_ci(y, pred, n_boot=500):
    rng_b = np.random.default_rng(SEED)
    vals = np.empty(n_boot)
    n = len(y)
    for i in range(n_boot):
        idx = rng_b.integers(0, n, size=n)
        vals[i] = score_pr(y[idx], pred[idx])
    return float(np.quantile(vals, 0.05)), float(np.quantile(vals, 0.95))

best_audit_single = max((score_pr(y_audit, audit_preds[e]), e) for e in EXPERTS)
print(f"Audit 90% CI ensemble: {bootstrap_ci(y_audit, p_audit_blend)}")
print(f"Audit 90% CI best single ({best_audit_single[1]}): {bootstrap_ci(y_audit, audit_preds[best_audit_single[1]])}")

## 16. Pseudo-labeling

В этом разделе я оставил возможность использовать псевдоразметку, то есть добавить в обучение часть тестовых объектов с наиболее уверенными предсказаниями модели. Идея в том, чтобы использовать не только размеченный `train`, но и часть информации из `test`, если модель достаточно уверена в своих прогнозах.

В текущей версии я этот подход не использую: `USE_PSEUDO_LABELS = False`. Во время экспериментов псевдоразметка не дала достаточно полезного и стабильного результата, поэтому в финальное решение её не включал.

Саму секцию я оставил в коде, чтобы при необходимости можно было быстро включить этот вариант и сравнить его с основной схемой. Поэтому сейчас этот блок фактически ничего не меняет в итоговом обучении.

In [ ]:
def load_pseudo_labels() -> Optional[pd.DataFrame]:
    if not USE_PSEUDO_LABELS:
        return None
    path = first_existing([
        Path("submission_v7_pseudo.csv"),
        Path("/content/submission_v7_pseudo.csv"),
        Path("artifacts/submission_v7_pseudo.csv"),
    ])
    if path is None:
        print("Pseudo-label file not found -> supervised final train")
        return None
    sub = pd.read_csv(path)
    if not {"cookie_id", "score"}.issubset(sub.columns):
        print("Pseudo-label file malformed -> ignored")
        return None
    bot = sub.loc[sub["score"] >= PSEUDO_BOT_THRESHOLD, ["cookie_id"]].copy()
    bot["target"] = 1
    hum = sub.loc[sub["score"] <= PSEUDO_HUMAN_THRESHOLD, ["cookie_id"]].copy()
    hum["target"] = 0
    pseudo = pd.concat([bot, hum], ignore_index=True).drop_duplicates("cookie_id")
    pseudo = pseudo[pseudo["cookie_id"].isin(set(test_meta["cookie_id"]))].copy()
    print(
        f"Pseudo labels: {len(pseudo)} rows, bots={int((pseudo.target==1).sum())}, humans={int((pseudo.target==0).sum())}"
    )
    return pseudo

pseudo = None

## 17. Финальное обучение и submission

После того как все параметры и веса ансамбля были выбраны, я перехожу к финальному обучению. Здесь уже используются все доступные размеченные данные из `train`, а `selection` и `audit` больше не выделяются отдельно, потому что этап настройки уже закончен.

Для каждого эксперта я обучаю несколько одинаковых моделей с разными `seed`: `42`, `123` и `777`. Затем предсказания этих трёх моделей усредняются. Это позволяет немного уменьшить влияние случайности при обучении и получить более стабильный прогноз.

После этого отдельно получаются предсказания для LightGBM, CatBoost и XGBoost. Их я объединяю с помощью уже найденного на `calibration OOF` rank-space blend. То есть веса и степень преобразования здесь уже не подбираются заново, а просто применяются к финальным предсказаниям.

В конце получаются итоговые предсказания для всех объектов `test`, и из них формируется `submission.csv`. Именно этот файл отправляется на проверку скрытой тестовой выборке.

In [ ]:
Xtest_ext = X_test_all[feature_cols_ext].copy()
Xtest_hyb = X_test_all[feature_cols_hybrid].copy()

pseudo = None

# Обучаемся на всем трейне
X_ext_final = X_train_all[feature_cols_ext].copy()
y_ext_final = y_train.copy()

sw_ext = None

X_hyb_final = X_train_all[feature_cols_hybrid].copy()

# Функция для весов
def make_recency_weights(meta_df: pd.DataFrame, mode: str) -> np.ndarray:
    if mode == "none":
        return np.ones(len(meta_df), dtype=np.float32)
    days = pd.to_datetime(meta_df["window_start_ts"]).dt.floor("D")
    age = (days.max() - days).dt.total_seconds() / 86400.0
    if mode == "linear_25":
        denom = max(float(age.max()), 1.0)
        w = 1.25 - 0.25 * (age / denom)
        return w.to_numpy(dtype=np.float32)
    if mode == "half_life_7d":
        return np.power(0.5, age.to_numpy(dtype=np.float64) / 7.0).astype(np.float32)
    raise ValueError(f"Unknown FINAL_RECENCY_MODE={mode!r}")
sw_final = make_recency_weights(train_meta, FINAL_RECENCY_MODE)

if FINAL_RECENCY_MODE == "none":
    sw_final_for_models = None
else:
    sw_final_for_models = sw_final

# Кодируем категории
def encode_xgb(train_frame: pd.DataFrame, test_frame: pd.DataFrame):
    tr = train_frame.copy()
    te = test_frame.copy()
    if "dominant_platform" in tr.columns:
        cats = pd.concat([
            tr["dominant_platform"].astype("string"),
            te["dominant_platform"].astype("string"),
        ]).fillna("missing").astype("category")
        tr["dominant_platform" ] = cats.iloc[:len(tr)].cat.codes.astype("int16")
        te["dominant_platform"] = cats.iloc[len(tr):].cat.codes.astype("int16")
    return tr, te

# Обучаем модели по разным сидам
def fit_seed_mean_lgb(params, n_estimators, Xtr, ytr, Xte, sample_weight=None, seeds=FINAL_SEEDS):
    preds = []
    for seed in seeds:
        m = make_lgb(params, n_estimators, ytr, seed=seed)
        m.fit(Xtr, ytr, sample_weight=sample_weight)
        preds.append(m.predict_proba(Xte)[:, 1])
    return np.mean(preds, axis=0), m

def fit_seed_mean_cat(params, iterations, Xtr, ytr, Xte, catcols, sample_weight=None, seeds=FINAL_SEEDS):
    preds = []
    last_model = None
    for seed in seeds:
        m = make_cat(params, iterations, ytr, catcols, seed=seed)
        m.fit(Xtr, ytr, sample_weight=sample_weight)
        preds.append(m.predict_proba(Xte)[:, 1])
        last_model = m
    return np.mean(preds, axis=0), last_model

def fit_seed_mean_xgb(params, n_estimators, Xtr, ytr, Xte, sample_weight=None, seeds=FINAL_SEEDS):
    preds = []
    last_model = None
    for seed in seeds:
        m = make_xgb(params, n_estimators, ytr, seed=seed)
        m.fit(Xtr, ytr, sample_weight=sample_weight, verbose=False)
        preds.append(m.predict_proba(Xte)[:, 1])
        last_model = m
    return np.mean(preds, axis=0), last_model

# категориальные признаки
X_ext_final_cb = prepare_cat(X_ext_final, feature_cols_ext)
Xtest_ext_cb = prepare_cat(Xtest_ext, feature_cols_ext)

X_hyb_final_model = X_hyb_final.copy()
Xtest_hyb_model = Xtest_hyb.copy()

if "dominant_platform" in X_hyb_final_model.columns:
    X_hyb_final_model["dominant_platform"] = X_hyb_final_model["dominant_platform"].astype("category")
    Xtest_hyb_model["dominant_platform"] = Xtest_hyb_model["dominant_platform"].astype("category")

for frame in [X_hyb_final_model, Xtest_hyb_model]:
    for c in frame.columns:
        if c != "dominant_platform" and frame[c].dtype == "object":
            frame[c] = pd.to_numeric(frame[c], errors="coerce")

# Запуск обучения
test_ext_lgb, final_ext_lgb = fit_seed_mean_lgb(
    best_ext_lgb_params, best_ext_lgb_n, X_ext_final, y_ext_final, Xtest_ext, sw_final_for_models
)

test_ext_cat, final_ext_cat = fit_seed_mean_cat(
    best_ext_cat_params, best_ext_cat_n, X_ext_final_cb, y_ext_final, Xtest_ext_cb,
    CAT_FEATURES, sw_final_for_models
)

test_hyb_lgb, final_hyb_lgb = fit_seed_mean_lgb(
    best_hybrid_lgb_params, best_hybrid_lgb_n, X_hyb_final_model, y_ext_final, Xtest_hyb_model,
    sw_final_for_models
)

Xxf, Xxt = encode_xgb(X_ext_final, Xtest_ext)

test_ext_xgb, final_ext_xgb = fit_seed_mean_xgb(
    best_ext_xgb_params, best_ext_xgb_n, Xxf, y_ext_final, Xxt, sw_final_for_models
)

print(f"Final seed ensemble: {FINAL_SEEDS}")
print(f"Final recency mode : {FINAL_RECENCY_MODE}")

def make_final_submission(ext_lgb_pred, ext_cat_pred, hyb_pred, ext_xgb_pred, path: Path, weights=None):
    preds = {
        "ext_lgb": ext_lgb_pred,
        "ext_cat": ext_cat_pred,
        "hybrid_lgb": hyb_pred,
        "ext_xgb": ext_xgb_pred,
    }
    use_weights = best_weights if weights is None else weights
    final_rank = np.zeros(len(test_meta), dtype=float)
    for e, w in zip(EXPERTS, use_weights):
         final_rank += w * np.power(rank01(preds[e]), best_power)
    out = pd.DataFrame({"cookie_id": test_meta["cookie_id"], "score": final_rank})
    out.to_csv(path, index=False)
    return out

submission_blend = make_final_submission(
    test_ext_lgb, test_ext_cat, test_hyb_lgb, test_ext_xgb,
    ARTIFACTS_DIR / "submission_v14_1_blend.csv",
)

# Отдельные сабмиты одиночных моделей
submission_ext_cat = pd.DataFrame({
    "cookie_id": test_meta["cookie_id"],
    "score": rank01(test_ext_cat),
})
submission_ext_cat.to_csv(ARTIFACTS_DIR / "submission_v14_1_extended_cat.csv", index=False)

submission_ext_lgb = pd.DataFrame({
    "cookie_id": test_meta["cookie_id"],
    "score": rank01(test_ext_lgb),
})
submission_ext_lgb.to_csv(ARTIFACTS_DIR / "submission_v14_1_extended_lgb.csv", index=False)

submission_xgb = pd.DataFrame({
    "cookie_id": test_meta["cookie_id"],
    "score": rank01(test_ext_xgb),
})
submission_xgb.to_csv(ARTIFACTS_DIR / "submission_v14_1_extended_xgb.csv", index=False)

cat_lgb_weights = {"ext_lgb": 0.40, "ext_cat": 0.60, "hybrid_lgb": 0.0, "ext_xgb": 0.0}
alt_weights = tuple(cat_lgb_weights[e] for e in EXPERTS)
submission_cat_lgb = make_final_submission(
    test_ext_lgb, test_ext_cat, test_hyb_lgb, test_ext_xgb,
    ARTIFACTS_DIR / "submission_v14_1_cat_lgb_40_60.csv",
    weights=alt_weights,
)

recommended_path = ARTIFACTS_DIR / "submission_v3.csv"
submission_blend.to_csv(recommended_path, index=False)

## Все, модель обучена, лучший score на скрытых тестах - 0.85